In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/feedback-prize-english-language-learning/sample_submission.csv
/kaggle/input/competitions/feedback-prize-english-language-learning/train.csv
/kaggle/input/competitions/feedback-prize-english-language-learning/test.csv


In [2]:
import pandas as pd
import numpy as np

DATA_DIR = '/kaggle/input/competitions/feedback-prize-english-language-learning'

train = pd.read_csv('/kaggle/input/competitions/feedback-prize-english-language-learning/train.csv')
test  = pd.read_csv('/kaggle/input/competitions/feedback-prize-english-language-learning/test.csv')
sub   = pd.read_csv('/kaggle/input/competitions/feedback-prize-english-language-learning/sample_submission.csv')

TARGETS = ['cohesion', 'syntax', 'vocabulary',
           'phraseology', 'grammar', 'conventions']

print('train:', train.shape)
print('test :', test.shape)
print('sub  :', sub.shape)
print()
print('train columns:', list(train.columns))
print('test columns :', list(test.columns))

train: (3911, 8)
test : (3, 2)
sub  : (3, 7)

train columns: ['text_id', 'full_text', 'cohesion', 'syntax', 'vocabulary', 'phraseology', 'grammar', 'conventions']
test columns : ['text_id', 'full_text']


## 1. First look at the data

In [3]:
print('--- missing values per column ---')
print(train.isnull().sum())

print('\n--- duplicate text_ids ---')
print(train['text_id'].duplicated().sum())

print('\n--- duplicate essays ---')
print(train['full_text'].duplicated().sum())

print('\n--- first 3 rows (targets only) ---')
print(train[['text_id'] + TARGETS].head(3))

print('\n--- unique values in each target ---')
for col in TARGETS:
    print(f'{col:12s}: {sorted(train[col].unique())}')

--- missing values per column ---
text_id        0
full_text      0
cohesion       0
syntax         0
vocabulary     0
phraseology    0
grammar        0
conventions    0
dtype: int64

--- duplicate text_ids ---
0

--- duplicate essays ---
0

--- first 3 rows (targets only) ---
        text_id  cohesion  syntax  vocabulary  phraseology  grammar  \
0  0016926B079C       3.5     3.5         3.0          3.0      4.0   
1  0022683E9EA5       2.5     2.5         3.0          2.0      2.0   
2  00299B378633       3.0     3.5         3.0          3.0      3.0   

   conventions  
0          3.0  
1          2.5  
2          2.5  

--- unique values in each target ---
cohesion    : [np.float64(1.0), np.float64(1.5), np.float64(2.0), np.float64(2.5), np.float64(3.0), np.float64(3.5), np.float64(4.0), np.float64(4.5), np.float64(5.0)]
syntax      : [np.float64(1.0), np.float64(1.5), np.float64(2.0), np.float64(2.5), np.float64(3.0), np.float64(3.5), np.float64(4.0), np.float64(4.5), np.float64(5

## 4. Target distributions

In [4]:
print('--- summary statistics ---')
print(train[TARGETS].describe().round(3))

print('\n--- distribution of each target (% of rows) ---')
dist = pd.DataFrame({
    col: train[col].value_counts(normalize=True).sort_index() * 100
    for col in TARGETS
}).round(1)
print(dist)

print('\n--- standard deviation per target ---')
print(train[TARGETS].std(ddof=0).round(4))

print('\n--- mean-prediction baseline (MCRMSE) ---')
print(round(train[TARGETS].std(ddof=0).mean(), 4))

--- summary statistics ---
       cohesion    syntax  vocabulary  phraseology   grammar  conventions
count  3911.000  3911.000    3911.000     3911.000  3911.000     3911.000
mean      3.127     3.028       3.236        3.117     3.033        3.081
std       0.663     0.644       0.583        0.656     0.700        0.671
min       1.000     1.000       1.000        1.000     1.000        1.000
25%       2.500     2.500       3.000        2.500     2.500        2.500
50%       3.000     3.000       3.000        3.000     3.000        3.000
75%       3.500     3.500       3.500        3.500     3.500        3.500
max       5.000     5.000       5.000        5.000     5.000        5.000

--- distribution of each target (% of rows) ---
     cohesion  syntax  vocabulary  phraseology  grammar  conventions
1.0       0.3     0.3         0.1          0.3      0.2          0.4
1.5       0.7     0.7         0.4          0.3      0.5          0.5
2.0       8.1    10.5         3.2          8.9     

## 5. Essay length

In [5]:
train['n_words'] = train['full_text'].str.split().str.len()
train['n_chars'] = train['full_text'].str.len()
train['n_paras'] = train['full_text'].str.count('\n\n') + 1

print('--- word count ---')
print(train['n_words'].describe(
    percentiles=[.05, .25, .5, .75, .9, .95, .99]).round(0))

print('\n--- char count ---')
print(train['n_chars'].describe(percentiles=[.5, .95]).round(0))

print('\n--- paragraph count ---')
print(train['n_paras'].describe(percentiles=[.5, .95]).round(1))

print('\n--- correlation of word count with each target ---')
for col in TARGETS:
    print(f'{col:12s}: {train["n_words"].corr(train[col]):+.3f}')

print('\n--- shortest 5 essays (words) ---')
print(train.nsmallest(5, 'n_words')[['text_id', 'n_words'] + TARGETS].to_string(index=False))

--- word count ---
count    3911.0
mean      430.0
std       192.0
min        14.0
5%        188.0
25%       294.0
50%       402.0
75%       526.0
90%       672.0
95%       788.0
99%      1100.0
max      1260.0
Name: n_words, dtype: float64

--- char count ---
count    3911.0
mean     2335.0
std      1033.0
min        82.0
50%      2173.0
95%      4288.0
max      6044.0
Name: n_chars, dtype: float64

--- paragraph count ---
count    3911.0
mean        5.5
std         3.1
min         1.0
50%         5.0
95%        10.0
max        52.0
Name: n_paras, dtype: float64

--- correlation of word count with each target ---
cohesion    : +0.219
syntax      : +0.188
vocabulary  : +0.271
phraseology : +0.214
grammar     : +0.080
conventions : +0.143

--- shortest 5 essays (words) ---
     text_id  n_words  cohesion  syntax  vocabulary  phraseology  grammar  conventions
F69C85F4C3CA       14       1.0     1.0         1.5          1.0      1.0          1.5
7835355C55D8       48       2.5     2.5    

## 6. Read the essays

In [6]:
train['mean_score'] = train[TARGETS].mean(axis=1)

def show(row, label):
    print('=' * 70)
    print(f'{label}   |   text_id: {row.text_id}   |   {row.n_words} words')
    print('scores:', {c: row[c] for c in TARGETS})
    print('-' * 70)
    print(row.full_text[:1200])
    if len(row.full_text) > 1200:
        print(f'\n... [truncated, {len(row.full_text)} chars total]')
    print()

low  = train.nsmallest(1, 'mean_score').iloc[0]
high = train.nlargest(1,  'mean_score').iloc[0]
mid  = train.iloc[(train['mean_score'] - 3.0).abs().argsort()[:1]].iloc[0]

show(low,  'LOW SCORER')
show(mid,  'MID SCORER')
show(high, 'HIGH SCORER')

LOW SCORER   |   text_id: 48EA282A4EAF   |   255 words
scores: {'cohesion': np.float64(1.0), 'syntax': np.float64(1.0), 'vocabulary': np.float64(1.0), 'phraseology': np.float64(1.0), 'grammar': np.float64(1.0), 'conventions': np.float64(1.0)}
----------------------------------------------------------------------
some student offer distance learning as an option for student to attend classes from homr by wat of online pr video conferencing. i think student would benefit form being able to attend classesfrom home. you are authorized take the electronic version of this you will taking this promptsome student offer distance learning as an option for student to attend classes from homr by wat of online pr video conferencing. some student offer distance learning as an option for student to attend classes from homr by wat of online pr video conferencing. some student offer distance learning as an option.

online pr video conferencing. the right view the prompt and teh checklist for writers vv

## 7. Target correlations

In [7]:
print('--- correlation matrix (Pearson) ---')
corr = train[TARGETS].corr()
print(corr.round(3))

print('\n--- summary ---')
vals = corr.values[np.triu_indices_from(corr.values, k=1)]
print(f'number of pairs      : {len(vals)}')
print(f'lowest correlation   : {vals.min():.3f}')
print(f'highest correlation  : {vals.max():.3f}')
print(f'mean correlation     : {vals.mean():.3f}')

print('\n--- how often do all 6 targets share the same value? ---')
same = (train[TARGETS].nunique(axis=1) == 1).mean()
print(f'{same*100:.1f}% of essays')

print('\n--- spread within an essay (max target - min target) ---')
spread = train[TARGETS].max(axis=1) - train[TARGETS].min(axis=1)
print(spread.value_counts(normalize=True).sort_index().round(3) * 100)

--- correlation matrix (Pearson) ---
             cohesion  syntax  vocabulary  phraseology  grammar  conventions
cohesion        1.000   0.695       0.666        0.690    0.639        0.666
syntax          0.695   1.000       0.681        0.725    0.710        0.700
vocabulary      0.666   0.681       1.000        0.735    0.655        0.664
phraseology     0.690   0.725       0.735        1.000    0.720        0.667
grammar         0.639   0.710       0.655        0.720    1.000        0.673
conventions     0.666   0.700       0.664        0.667    0.673        1.000

--- summary ---
number of pairs      : 15
lowest correlation   : 0.639
highest correlation  : 0.735
mean correlation     : 0.686

--- how often do all 6 targets share the same value? ---
1.7% of essays

--- spread within an essay (max target - min target) ---
0.0     1.7
0.5    32.5
1.0    56.4
1.5     9.1
2.0     0.3
Name: proportion, dtype: float64


In [8]:
!pip install -q iterative-stratification

In [9]:
from sklearn.model_selection import KFold
from iterstrat.ml_stratifiers import MultilabelStratifiedKFold

SEED = 42
N_FOLDS = 5

# --- method A: plain random KFold ---
fold_plain = np.zeros(len(train), dtype=int)
kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
for i, (_, val_idx) in enumerate(kf.split(train)):
    fold_plain[val_idx] = i

# --- method B: MultilabelStratifiedKFold, raw scores ---
fold_strat = np.zeros(len(train), dtype=int)
mskf = MultilabelStratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
for i, (_, val_idx) in enumerate(mskf.split(train, train[TARGETS])):
    fold_strat[val_idx] = i

# --- compare: how different are the target means across folds? ---
def imbalance(fold_array):
    means = train.groupby(fold_array)[TARGETS].mean()
    return (means.max() - means.min())

print('--- METHOD A: plain KFold ---')
print(imbalance(fold_plain).round(4).to_string())
print(f'average: {imbalance(fold_plain).mean():.4f}')

print('\n--- METHOD B: MultilabelStratifiedKFold (raw scores) ---')
print(imbalance(fold_strat).round(4).to_string())
print(f'average: {imbalance(fold_strat).mean():.4f}')

--- METHOD A: plain KFold ---
cohesion       0.0083
syntax         0.0307
vocabulary     0.0412
phraseology    0.0390
grammar        0.0448
conventions    0.0332
average: 0.0329

--- METHOD B: MultilabelStratifiedKFold (raw scores) ---
cohesion       0.0838
syntax         0.0965
vocabulary     0.0531
phraseology    0.0767
grammar        0.1164
conventions    0.0812
average: 0.0846


## 9. Cross-validation: fixed version

In [10]:
# convert each target into 9 binary columns -> 6 x 9 = 54 columns
y_onehot = pd.get_dummies(train[TARGETS].astype(str)).values.astype(int)
print('one-hot shape:', y_onehot.shape)
print('unique values inside:', np.unique(y_onehot))

# --- method C: MultilabelStratifiedKFold with one-hot ---
fold_onehot = np.zeros(len(train), dtype=int)
mskf = MultilabelStratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
for i, (_, val_idx) in enumerate(mskf.split(train, y_onehot)):
    fold_onehot[val_idx] = i

# --- compare all three methods ---
print('\n--- fold imbalance (lower = better) ---')
result = pd.DataFrame({
    'A_plain_kfold'   : imbalance(fold_plain),
    'B_mskf_raw'      : imbalance(fold_strat),
    'C_mskf_onehot'   : imbalance(fold_onehot),
}).round(4)
print(result.to_string())
print('\naverage:')
print(result.mean().round(4).to_string())

one-hot shape: (3911, 54)
unique values inside: [0 1]

--- fold imbalance (lower = better) ---
             A_plain_kfold  B_mskf_raw  C_mskf_onehot
cohesion            0.0083      0.0838         0.0128
syntax              0.0307      0.0965         0.0177
vocabulary          0.0412      0.0531         0.0084
phraseology         0.0390      0.0767         0.0083
grammar             0.0448      0.1164         0.0170
conventions         0.0332      0.0812         0.0070

average:
A_plain_kfold    0.0329
B_mskf_raw       0.0846
C_mskf_onehot    0.0119


## 10. Save folds

In [11]:
train['fold'] = fold_onehot

print('--- rows per fold ---')
print(train['fold'].value_counts().sort_index().to_string())

print('\n--- target means per fold ---')
print(train.groupby('fold')[TARGETS].mean().round(4).to_string())

# drop the temporary EDA columns so the saved file stays clean
train_out = train.drop(columns=['n_words', 'n_chars', 'n_paras', 'mean_score'])

train_out.to_csv('train_5folds.csv', index=False)
print('\nsaved -> train_5folds.csv')
print('columns:', list(train_out.columns))

--- rows per fold ---
fold
0    781
1    787
2    785
3    780
4    778

--- target means per fold ---
      cohesion  syntax  vocabulary  phraseology  grammar  conventions
fold                                                                 
0       3.1216  3.0262      3.2324       3.1165   3.0333       3.0768
1       3.1264  3.0311      3.2351       3.1194   3.0375       3.0839
2       3.1344  3.0363      3.2408       3.1191   3.0382       3.0828
3       3.1237  3.0288      3.2340       3.1179   3.0340       3.0814
4       3.1292  3.0186      3.2365       3.1112   3.0212       3.0803

saved -> train_5folds.csv
columns: ['text_id', 'full_text', 'cohesion', 'syntax', 'vocabulary', 'phraseology', 'grammar', 'conventions', 'fold']


## 11. The metric: MCRMSE

In [12]:
def mcrmse(y_true, y_pred, per_column=False):
    col_rmse = np.sqrt(np.mean((y_true - y_pred) ** 2, axis=0))
    if per_column:
        return col_rmse.mean(), col_rmse
    return col_rmse.mean()


y = train[TARGETS].values

print('TESTING THE METRIC FUNCTION')
print('=' * 45)

# test 1
t1 = mcrmse(y, y)
print(f'1. Perfect prediction')
print(f'   expected 0.0   got {t1:.4f}   {"PASS" if t1 == 0 else "FAIL"}')

# test 2
t2 = mcrmse(y, y + 1.0)
print(f'\n2. Every prediction wrong by exactly 1.0')
print(f'   expected 1.0   got {t2:.4f}   {"PASS" if abs(t2-1) < 1e-9 else "FAIL"}')

# test 3
mean_pred = np.tile(y.mean(axis=0), (len(y), 1))
t3 = mcrmse(y, mean_pred)
t3_check = train[TARGETS].std(ddof=0).mean()
print(f'\n3. Predict the mean  ==  standard deviation?')
print(f'   metric says {t3:.4f}   std says {t3_check:.4f}   '
      f'{"PASS" if abs(t3-t3_check) < 1e-9 else "FAIL"}')

print('=' * 45)

TESTING THE METRIC FUNCTION
1. Perfect prediction
   expected 0.0   got 0.0000   PASS

2. Every prediction wrong by exactly 1.0
   expected 1.0   got 1.0000   PASS

3. Predict the mean  ==  standard deviation?
   metric says 0.6528   std says 0.6528   PASS


In [13]:
_, cols = mcrmse(y, mean_pred, per_column=True)

print('BASELINE: predict the mean')
print('=' * 40)
for name, val in zip(TARGETS, cols):
    print(f'  {name:<14} {val:.4f}')
print('-' * 40)
print(f'  {"MCRMSE":<14} {cols.mean():.4f}')
print('=' * 40)

BASELINE: predict the mean
  cohesion       0.6625
  syntax         0.6443
  vocabulary     0.5831
  phraseology    0.6559
  grammar        0.6998
  conventions    0.6714
----------------------------------------
  MCRMSE         0.6528


## 13. Baseline B: TF-IDF + Ridge

In [14]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import Ridge

# --- look at what TF-IDF produces, on a tiny example first ---
sample = [
    "I think students learn better at home.",
    "Students dont learn good at home.",
    "Distance learning helps students a lot."
]

vec = TfidfVectorizer()
X = vec.fit_transform(sample)

print('vocabulary the model built:')
print(vec.get_feature_names_out())

print(f'\nmatrix shape: {X.shape}   (3 documents, {X.shape[1]} words)')

print('\nthe matrix as numbers:')
print(pd.DataFrame(X.toarray().round(2),
                   columns=vec.get_feature_names_out(),
                   index=['doc1', 'doc2', 'doc3']).to_string())

vocabulary the model built:
['at' 'better' 'distance' 'dont' 'good' 'helps' 'home' 'learn' 'learning'
 'lot' 'students' 'think']

matrix shape: (3, 12)   (3 documents, 12 words)

the matrix as numbers:
        at  better  distance  dont  good  helps  home  learn  learning   lot  students  think
doc1  0.38    0.49      0.00  0.00  0.00   0.00  0.38   0.38      0.00  0.00      0.29   0.49
doc2  0.38    0.00      0.00  0.49  0.49   0.00  0.38   0.38      0.00  0.00      0.29   0.00
doc3  0.00    0.00      0.48  0.00  0.00   0.48  0.00   0.00      0.48  0.48      0.28   0.00


## 14. TF-IDF step 1: plain defaults

In [15]:
def run_tfidf_ridge(word_params, char_params=None, alpha=10.0, verbose=True):
    """Train TF-IDF + Ridge with 5-fold CV. Returns (score, oof, n_features)."""
    oof = np.zeros((len(train), len(TARGETS)))
    n_feat = 0

    for f in range(N_FOLDS):
        tr_idx = train.index[train['fold'] != f]
        va_idx = train.index[train['fold'] == f]
        txt_tr = train.loc[tr_idx, 'full_text']
        txt_va = train.loc[va_idx, 'full_text']

        wv = TfidfVectorizer(**word_params)
        Xtr = wv.fit_transform(txt_tr)
        Xva = wv.transform(txt_va)

        if char_params is not None:
            cv = TfidfVectorizer(**char_params)
            Xtr = hstack([Xtr, cv.fit_transform(txt_tr)]).tocsr()
            Xva = hstack([Xva, cv.transform(txt_va)]).tocsr()

        n_feat = Xtr.shape[1]

        model = Ridge(alpha=alpha)
        model.fit(Xtr, train.loc[tr_idx, TARGETS].values)
        oof[va_idx] = model.predict(Xva)

    oof = np.clip(oof, 1.0, 5.0)
    score = mcrmse(train[TARGETS].values, oof)
    if verbose:
        print(f'  features: {n_feat:>7,}    MCRMSE: {score:.4f}')
    return score, oof, n_feat


from scipy.sparse import hstack

results = {}

print('STEP 1: plain defaults (single words, lowercased)')
results['1_default'], oof1, _ = run_tfidf_ridge(
    word_params=dict(ngram_range=(1, 1), min_df=3, max_features=50000)
)

STEP 1: plain defaults (single words, lowercased)
  features:   6,221    MCRMSE: 0.5721


## 15. TF-IDF step 2: keep capitalisation

In [16]:
print('STEP 2: lowercase=False (keep capitalisation)')
results['2_no_lowercase'], oof2, _ = run_tfidf_ridge(
    word_params=dict(ngram_range=(1, 1), min_df=3, max_features=50000,
                     lowercase=False)
)

print(f'\nstep 1 : {results["1_default"]:.4f}')
print(f'step 2 : {results["2_no_lowercase"]:.4f}')
print(f'change : {results["1_default"] - results["2_no_lowercase"]:+.4f}')

STEP 2: lowercase=False (keep capitalisation)
  features:   6,930    MCRMSE: 0.5680

step 1 : 0.5721
step 2 : 0.5680
change : +0.0041


## 16. TF-IDF step 3: add word pairs

In [17]:
print('STEP 3: ngram_range=(1,2) — single words + word pairs')
results['3_bigrams'], oof3, _ = run_tfidf_ridge(
    word_params=dict(ngram_range=(1, 2), min_df=3, max_features=50000,
                     lowercase=False)
)

print(f'\nstep 1 (defaults)      : {results["1_default"]:.4f}')
print(f'step 2 (keep capitals) : {results["2_no_lowercase"]:.4f}')
print(f'step 3 (+ word pairs)  : {results["3_bigrams"]:.4f}')

STEP 3: ngram_range=(1,2) — single words + word pairs
  features:  50,000    MCRMSE: 0.5762

step 1 (defaults)      : 0.5721
step 2 (keep capitals) : 0.5680
step 3 (+ word pairs)  : 0.5762


## 17. Diagnose: was the cap the problem?

In [18]:
print('STEP 3b: word pairs, cap raised to 200,000')
results['3b_bigrams_bigcap'], oof3b, _ = run_tfidf_ridge(
    word_params=dict(ngram_range=(1, 2), min_df=3, max_features=200000,
                     lowercase=False)
)

print('\nSTEP 3c: word pairs, no cap at all')
results['3c_bigrams_nocap'], oof3c, _ = run_tfidf_ridge(
    word_params=dict(ngram_range=(1, 2), min_df=3, max_features=None,
                     lowercase=False)
)

print('\n' + '=' * 48)
print(f'{"setting":<32}{"MCRMSE":>10}')
print('-' * 48)
print(f'{"single words only":<32}{results["2_no_lowercase"]:>10.4f}')
print(f'{"+ pairs, cap 50k":<32}{results["3_bigrams"]:>10.4f}')
print(f'{"+ pairs, cap 200k":<32}{results["3b_bigrams_bigcap"]:>10.4f}')
print(f'{"+ pairs, no cap":<32}{results["3c_bigrams_nocap"]:>10.4f}')
print('=' * 48)

STEP 3b: word pairs, cap raised to 200,000
  features:  57,790    MCRMSE: 0.5768

STEP 3c: word pairs, no cap at all
  features:  57,790    MCRMSE: 0.5768

setting                             MCRMSE
------------------------------------------------
single words only                   0.5680
+ pairs, cap 50k                    0.5762
+ pairs, cap 200k                   0.5768
+ pairs, no cap                     0.5768


## 18. TF-IDF step 4: add character n-grams

In [19]:
print('STEP 4: single words + character chunks (3-5 letters)')
results['4_char'], oof4, _ = run_tfidf_ridge(
    word_params=dict(ngram_range=(1, 1), min_df=3, max_features=50000,
                     lowercase=False),
    char_params=dict(analyzer='char_wb', ngram_range=(3, 5), min_df=3,
                     max_features=50000, lowercase=False)
)

print('\n' + '=' * 48)
print(f'{"setting":<32}{"MCRMSE":>10}')
print('-' * 48)
print(f'{"predict the mean":<32}{0.6528:>10.4f}')
print(f'{"words, defaults":<32}{results["1_default"]:>10.4f}')
print(f'{"words, keep capitals":<32}{results["2_no_lowercase"]:>10.4f}')
print(f'{"+ word pairs":<32}{results["3_bigrams"]:>10.4f}')
print(f'{"+ character chunks":<32}{results["4_char"]:>10.4f}')
print('=' * 48)

STEP 4: single words + character chunks (3-5 letters)
  features:  54,172    MCRMSE: 0.5480

setting                             MCRMSE
------------------------------------------------
predict the mean                    0.6528
words, defaults                     0.5721
words, keep capitals                0.5680
+ word pairs                        0.5762
+ character chunks                  0.5480


## 19. Tune Ridge alpha

In [20]:
best_word = dict(ngram_range=(1, 1), min_df=3, max_features=50000, lowercase=False)
best_char = dict(analyzer='char_wb', ngram_range=(3, 5), min_df=3,
                 max_features=50000, lowercase=False)

alphas = [0.1, 1.0, 10.0, 50.0, 200.0, 1000.0]
alpha_scores = {}

for a in alphas:
    print(f'alpha = {a}')
    s, _, _ = run_tfidf_ridge(best_word, best_char, alpha=a)
    alpha_scores[a] = s

print('\n' + '=' * 34)
print(f'{"alpha":>10}{"MCRMSE":>14}')
print('-' * 34)
for a, s in alpha_scores.items():
    mark = '  <-- best' if s == min(alpha_scores.values()) else ''
    print(f'{a:>10}{s:>14.4f}{mark}')
print('=' * 34)

alpha = 0.1
  features:  54,172    MCRMSE: 0.6123
alpha = 1.0
  features:  54,172    MCRMSE: 0.5402
alpha = 10.0
  features:  54,172    MCRMSE: 0.5480
alpha = 50.0
  features:  54,172    MCRMSE: 0.5938
alpha = 200.0
  features:  54,172    MCRMSE: 0.6299
alpha = 1000.0
  features:  54,172    MCRMSE: 0.6473

     alpha        MCRMSE
----------------------------------
       0.1        0.6123
       1.0        0.5402  <-- best
      10.0        0.5480
      50.0        0.5938
     200.0        0.6299
    1000.0        0.6473


In [21]:
fine_alphas = [0.3, 0.5, 1.0, 2.0, 3.0, 5.0]
fine_scores = {}

for a in fine_alphas:
    print(f'alpha = {a}')
    s, _, _ = run_tfidf_ridge(best_word, best_char, alpha=a)
    fine_scores[a] = s

print('\n' + '=' * 34)
print(f'{"alpha":>10}{"MCRMSE":>14}')
print('-' * 34)
best_a = min(fine_scores, key=fine_scores.get)
for a, s in fine_scores.items():
    mark = '  <-- best' if a == best_a else ''
    print(f'{a:>10}{s:>14.4f}{mark}')
print('=' * 34)
print(f'\nchosen alpha: {best_a}')

alpha = 0.3
  features:  54,172    MCRMSE: 0.5693
alpha = 0.5
  features:  54,172    MCRMSE: 0.5542
alpha = 1.0
  features:  54,172    MCRMSE: 0.5402
alpha = 2.0
  features:  54,172    MCRMSE: 0.5339
alpha = 3.0
  features:  54,172    MCRMSE: 0.5338
alpha = 5.0
  features:  54,172    MCRMSE: 0.5372

     alpha        MCRMSE
----------------------------------
       0.3        0.5693
       0.5        0.5542
       1.0        0.5402
       2.0        0.5339
       3.0        0.5338  <-- best
       5.0        0.5372

chosen alpha: 3.0


## 20. Tune min_df

In [22]:
min_dfs = [1, 2, 3, 5, 10, 20]
mindf_scores = {}

for m in min_dfs:
    print(f'min_df = {m}')
    s, _, _ = run_tfidf_ridge(
        word_params=dict(ngram_range=(1, 1), min_df=m, max_features=50000,
                         lowercase=False),
        char_params=dict(analyzer='char_wb', ngram_range=(3, 5), min_df=m,
                         max_features=50000, lowercase=False),
        alpha=3.0
    )
    mindf_scores[m] = s

print('\n' + '=' * 34)
print(f'{"min_df":>10}{"MCRMSE":>14}')
print('-' * 34)
best_m = min(mindf_scores, key=mindf_scores.get)
for m, s in mindf_scores.items():
    mark = '  <-- best' if m == best_m else ''
    print(f'{m:>10}{s:>14.4f}{mark}')
print('=' * 34)

min_df = 1
  features:  71,439    MCRMSE: 0.5337
min_df = 2
  features:  59,503    MCRMSE: 0.5336
min_df = 3
  features:  54,172    MCRMSE: 0.5338
min_df = 5
  features:  40,534    MCRMSE: 0.5341
min_df = 10
  features:  28,461    MCRMSE: 0.5344
min_df = 20
  features:  20,406    MCRMSE: 0.5354

    min_df        MCRMSE
----------------------------------
         1        0.5337
         2        0.5336  <-- best
         3        0.5338
         5        0.5341
        10        0.5344
        20        0.5354


# Checking bigrams again with changing alpha

In [23]:
print('Re-testing bigrams with tuned alpha')
print('=' * 46)
for a in [1.0, 3.0, 10.0]:
    print(f'\nbigrams + chars, alpha={a}')
    run_tfidf_ridge(
        word_params=dict(ngram_range=(1, 2), min_df=3, max_features=200000,
                         lowercase=False),
        char_params=dict(analyzer='char_wb', ngram_range=(3, 5), min_df=3,
                         max_features=50000, lowercase=False),
        alpha=a
    )

print('\nfor comparison:')
print(f'  words only + chars, alpha=3.0 : 0.5338')

Re-testing bigrams with tuned alpha

bigrams + chars, alpha=1.0
  features: 105,032    MCRMSE: 0.5303

bigrams + chars, alpha=3.0
  features: 105,032    MCRMSE: 0.5306

bigrams + chars, alpha=10.0
  features: 105,032    MCRMSE: 0.5494

for comparison:
  words only + chars, alpha=3.0 : 0.5338


# Checking for other non-linear models

In [24]:
from sklearn.svm import LinearSVR
from sklearn.linear_model import SGDRegressor
from sklearn.multioutput import MultiOutputRegressor
from sklearn.decomposition import TruncatedSVD
import lightgbm as lgb

W = dict(ngram_range=(1,1), min_df=3, max_features=50000, lowercase=False)
C = dict(analyzer='char_wb', ngram_range=(3,5), min_df=3,
         max_features=50000, lowercase=False)

def run_model(make_model, use_svd=False, n_comp=300, tag=''):
    oof = np.zeros((len(train), len(TARGETS)))
    for f in range(N_FOLDS):
        tr = train.index[train['fold'] != f]
        va = train.index[train['fold'] == f]
        wv, cv = TfidfVectorizer(**W), TfidfVectorizer(**C)
        Xtr = hstack([wv.fit_transform(train.loc[tr,'full_text']),
                      cv.fit_transform(train.loc[tr,'full_text'])]).tocsr()
        Xva = hstack([wv.transform(train.loc[va,'full_text']),
                      cv.transform(train.loc[va,'full_text'])]).tocsr()
        if use_svd:
            svd = TruncatedSVD(n_components=n_comp, random_state=SEED)
            Xtr = svd.fit_transform(Xtr)
            Xva = svd.transform(Xva)
        m = make_model()
        m.fit(Xtr, train.loc[tr, TARGETS].values)
        oof[va] = m.predict(Xva)
    oof = np.clip(oof, 1.0, 5.0)
    s = mcrmse(train[TARGETS].values, oof)
    print(f'{tag:<38} {s:.4f}')
    return s

print(f'{"model":<38} {"MCRMSE"}')
print('-' * 48)
print(f'{"Ridge (our baseline)":<38} 0.5338')

run_model(lambda: MultiOutputRegressor(LinearSVR(C=0.5, max_iter=5000, random_state=SEED)),
          tag='LinearSVR (sparse)')

run_model(lambda: Ridge(alpha=3.0), use_svd=True, n_comp=300,
          tag='SVD-300 + Ridge')

run_model(lambda: MultiOutputRegressor(
              lgb.LGBMRegressor(n_estimators=500, learning_rate=0.05,
                                num_leaves=31, verbose=-1, random_state=SEED)),
          use_svd=True, n_comp=300, tag='SVD-300 + LightGBM')

model                                  MCRMSE
------------------------------------------------
Ridge (our baseline)                   0.5338
LinearSVR (sparse)                     0.5550
SVD-300 + Ridge                        0.5402


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/v

SVD-300 + LightGBM                     0.5692


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/v

np.float64(0.5692494900726716)

# alpha tuning for pairs+chars

In [25]:
BIG_W = dict(ngram_range=(1, 2), min_df=3, max_features=200000, lowercase=False)
BIG_C = dict(analyzer='char_wb', ngram_range=(3, 5), min_df=3,
             max_features=50000, lowercase=False)

print('ALPHA SWEEP (bigram config, 105k features)')
print('-' * 42)
alpha2 = {}
for a in [0.3, 0.5, 0.7, 1.0, 1.5, 2.0]:
    print(f'alpha = {a}')
    s, _, _ = run_tfidf_ridge(BIG_W, BIG_C, alpha=a)
    alpha2[a] = s

best_a2 = min(alpha2, key=alpha2.get)
print('\n' + '=' * 34)
print(f'{"alpha":>10}{"MCRMSE":>14}')
print('-' * 34)
for a, s in alpha2.items():
    print(f'{a:>10}{s:>14.4f}' + ('  <-- best' if a == best_a2 else ''))
print('=' * 34)
print(f'best alpha: {best_a2}')

ALPHA SWEEP (bigram config, 105k features)
------------------------------------------
alpha = 0.3
  features: 105,032    MCRMSE: 0.5421
alpha = 0.5
  features: 105,032    MCRMSE: 0.5363
alpha = 0.7
  features: 105,032    MCRMSE: 0.5330
alpha = 1.0
  features: 105,032    MCRMSE: 0.5303
alpha = 1.5
  features: 105,032    MCRMSE: 0.5288
alpha = 2.0
  features: 105,032    MCRMSE: 0.5288

     alpha        MCRMSE
----------------------------------
       0.3        0.5421
       0.5        0.5363
       0.7        0.5330
       1.0        0.5303
       1.5        0.5288  <-- best
       2.0        0.5288
best alpha: 1.5


# more alpha tuning for pairs+chars

In [26]:
print('ALPHA SWEEP — extending upward')
print('-' * 42)
alpha3 = dict(alpha2)   # keep previous results
for a in [3.0, 5.0, 8.0]:
    print(f'alpha = {a}')
    s, _, _ = run_tfidf_ridge(BIG_W, BIG_C, alpha=a)
    alpha3[a] = s

best_a3 = min(alpha3, key=alpha3.get)
print('\n' + '=' * 34)
print(f'{"alpha":>10}{"MCRMSE":>14}')
print('-' * 34)
for a in sorted(alpha3):
    print(f'{a:>10}{alpha3[a]:>14.4f}' + ('  <-- best' if a == best_a3 else ''))
print('=' * 34)
print(f'best alpha: {best_a3}')

ALPHA SWEEP — extending upward
------------------------------------------
alpha = 3.0
  features: 105,032    MCRMSE: 0.5306
alpha = 5.0
  features: 105,032    MCRMSE: 0.5361
alpha = 8.0
  features: 105,032    MCRMSE: 0.5444

     alpha        MCRMSE
----------------------------------
       0.3        0.5421
       0.5        0.5363
       0.7        0.5330
       1.0        0.5303
       1.5        0.5288  <-- best
       2.0        0.5288
       3.0        0.5306
       5.0        0.5361
       8.0        0.5444
best alpha: 1.5


# min_df tuning

In [27]:
print('MIN_DF SWEEP (bigram config, alpha=2.0)')
print('-' * 42)
mindf2 = {}
for m in [1, 2, 3, 5, 10]:
    print(f'min_df = {m}')
    s, _, _ = run_tfidf_ridge(
        word_params=dict(ngram_range=(1, 2), min_df=m, max_features=200000,
                         lowercase=False),
        char_params=dict(analyzer='char_wb', ngram_range=(3, 5), min_df=m,
                         max_features=50000, lowercase=False),
        alpha=2.0
    )
    mindf2[m] = s

best_m2 = min(mindf2, key=mindf2.get)
print('\n' + '=' * 34)
print(f'{"min_df":>10}{"MCRMSE":>14}')
print('-' * 34)
for m, s in mindf2.items():
    print(f'{m:>10}{s:>14.4f}' + ('  <-- best' if m == best_m2 else ''))
print('=' * 34)
print(f'best min_df: {best_m2}')

MIN_DF SWEEP (bigram config, alpha=2.0)
------------------------------------------
min_df = 1
  features: 250,000    MCRMSE: 0.5297
min_df = 2
  features: 141,721    MCRMSE: 0.5290
min_df = 3
  features: 105,032    MCRMSE: 0.5288
min_df = 5
  features:  70,173    MCRMSE: 0.5292
min_df = 10
  features:  43,161    MCRMSE: 0.5296

    min_df        MCRMSE
----------------------------------
         1        0.5297
         2        0.5290
         3        0.5288  <-- best
         5        0.5292
        10        0.5296
best min_df: 3


## 22. Final TF-IDF model — save OOF

In [28]:
FINAL_W = dict(ngram_range=(1, 2), min_df=3, max_features=200000, lowercase=False)
FINAL_C = dict(analyzer='char_wb', ngram_range=(3, 5), min_df=3,
               max_features=50000, lowercase=False)
FINAL_ALPHA = 2.0

score_tfidf, oof_tfidf, nfeat = run_tfidf_ridge(FINAL_W, FINAL_C,
                                                alpha=FINAL_ALPHA, verbose=False)
_, cols = mcrmse(train[TARGETS].values, oof_tfidf, per_column=True)

print('FINAL TF-IDF + RIDGE')
print('=' * 40)
for name, val in zip(TARGETS, cols):
    print(f'  {name:<14} {val:.4f}')
print('-' * 40)
print(f'  {"MCRMSE":<14} {score_tfidf:.4f}')
print(f'  {"features":<14} {nfeat:,}')
print('=' * 40)

np.save('oof_tfidf_ridge.npy', oof_tfidf)
print('\nsaved -> oof_tfidf_ridge.npy')
print('shape:', oof_tfidf.shape)

FINAL TF-IDF + RIDGE
  cohesion       0.5463
  syntax         0.5191
  vocabulary     0.4685
  phraseology    0.5273
  grammar        0.5700
  conventions    0.5415
----------------------------------------
  MCRMSE         0.5288
  features       105,032

saved -> oof_tfidf_ridge.npy
shape: (3911, 6)


## 23. Install feature engineering libraries

In [29]:
!pip install -q textstat lexicalrichness textdescriptives
!python -m spacy download en_core_web_sm

print('done')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.8/97.8 kB 3.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 177.1/177.1 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 254.3/254.3 kB 18.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 68.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 64.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.5/98.5 MB 18.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.39.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
kaggle-environments 1.29.3 requires numpy>=2.0, but yo

In [30]:
import numpy as np
import pandas as pd
import spacy
import textstat
import textdescriptives as td
from lexicalrichness import LexicalRichness

print('numpy      :', np.__version__)
print('pandas     :', pd.__version__)
print('spacy      :', spacy.__version__)

nlp = spacy.load('en_core_web_sm')
doc = nlp("The students learn quickly. They study hard every day.")

print('\nsentences  :', [s.text for s in doc.sents])
print('POS tags   :', [(t.text, t.pos_) for t in doc])
print('\nspaCy model loaded OK')

numpy      : 2.0.2
pandas     : 2.3.3
spacy      : 3.8.14

sentences  : ['The students learn quickly.', 'They study hard every day.']
POS tags   : [('The', 'DET'), ('students', 'NOUN'), ('learn', 'VERB'), ('quickly', 'ADV'), ('.', 'PUNCT'), ('They', 'PRON'), ('study', 'VERB'), ('hard', 'ADV'), ('every', 'DET'), ('day', 'NOUN'), ('.', 'PUNCT')]

spaCy model loaded OK


## 24. Run spaCy on all essays (once)

In [31]:
import time

nlp = spacy.load('en_core_web_sm')

texts = train['full_text'].tolist()

t0 = time.time()
docs = list(nlp.pipe(texts, batch_size=50, n_process=1))
print(f'processed {len(docs)} essays in {time.time()-t0:.1f} seconds')

d = docs[0]
print(f'\nfirst essay: {len(d)} tokens, {len(list(d.sents))} sentences')
print('first 15 tokens with tags:')
for t in d[:15]:
    print(f'  {t.text:<15} {t.pos_:<8} dep={t.dep_:<12} head={t.head.text}')

processed 3911 essays in 163.0 seconds

first essay: 292 tokens, 17 sentences
first 15 tokens with tags:
  I               PRON     dep=nsubj        head=think
  think           VERB     dep=ROOT         head=think
  that            SCONJ    dep=mark         head=benefit
  students        NOUN     dep=nsubj        head=benefit
  would           AUX      dep=aux          head=benefit
  benefit         VERB     dep=ccomp        head=think
  from            ADP      dep=prep         head=benefit
  learning        VERB     dep=pcomp        head=from
  at              ADP      dep=prep         head=learning
  home            NOUN     dep=pobj         head=at
  ,               PUNCT    dep=punct        head=benefit
  because         SCONJ    dep=mark         head=have
  they            PRON     dep=nsubj        head=have
  wo              AUX      dep=aux          head=have
  nt              PART     dep=neg          head=have


## 25. Feature group 1 — descriptive

In [32]:
def descriptive_features(doc, text):
    words = [t for t in doc if not t.is_punct and not t.is_space]
    sents = list(doc.sents)
    sent_lens = [len([t for t in s if not t.is_punct and not t.is_space]) for s in sents]
    sent_lens = [x for x in sent_lens if x > 0]
    word_lens = [len(t.text) for t in words]

    f = {}
    f['n_chars']     = len(text)
    f['n_words']     = len(words)
    f['n_sents']     = len(sent_lens)
    f['n_paras']     = text.count('\n\n') + 1
    f['chars_per_word'] = np.mean(word_lens) if word_lens else 0
    f['words_per_sent'] = np.mean(sent_lens) if sent_lens else 0
    f['sent_len_std']   = np.std(sent_lens)  if sent_lens else 0
    f['sent_len_max']   = max(sent_lens)     if sent_lens else 0
    f['sent_len_min']   = min(sent_lens)     if sent_lens else 0
    f['words_per_para'] = len(words) / max(f['n_paras'], 1)
    f['sents_per_para'] = len(sent_lens) / max(f['n_paras'], 1)
    return f


feat_desc = pd.DataFrame([descriptive_features(d, t) for d, t in zip(docs, texts)])
print('shape:', feat_desc.shape)
print()
print(feat_desc.describe().round(2).to_string())

print('\n--- correlation with each target ---')
corr_desc = pd.DataFrame(
    {c: feat_desc.corrwith(train[c]) for c in TARGETS}
).round(3)
print(corr_desc.to_string())

shape: (3911, 11)

       n_chars  n_words  n_sents  n_paras  chars_per_word  words_per_sent  sent_len_std  sent_len_max  sent_len_min  words_per_para  sents_per_para
count  3911.00  3911.00  3911.00  3911.00         3911.00         3911.00       3911.00       3911.00       3911.00         3911.00         3911.00
mean   2334.52   437.04    19.13     5.50            4.20           27.38         14.05         58.77          9.31           97.49            4.18
std    1033.41   194.75    10.38     3.13            0.28           18.69         11.30         36.40         13.31           74.29            3.19
min      82.00    14.00     1.00     1.00            3.27            6.76          0.00         12.00          1.00            6.55            0.20
25%    1597.00   299.00    11.00     4.00            4.01           18.07          7.61         37.00          5.00           59.54            2.25
50%    2173.00   408.00    18.00     5.00            4.18           22.60         10.73      

In [33]:
from lexicalrichness import LexicalRichness

def lexical_features(doc):
    words = [t.text.lower() for t in doc if not t.is_punct and not t.is_space]
    lemmas = [t.lemma_.lower() for t in doc if not t.is_punct and not t.is_space]
    n = len(words)
    f = {}

    f['ttr']       = len(set(words)) / n if n else 0
    f['lemma_ttr'] = len(set(lemmas)) / n if n else 0
    f['root_ttr']  = len(set(words)) / np.sqrt(n) if n else 0
    f['log_ttr']   = np.log(len(set(words))) / np.log(n) if n > 1 else 0

    try:
        lex = LexicalRichness(' '.join(words))
        f['mtld'] = lex.mtld(threshold=0.72) if n >= 50 else np.nan
        f['hdd']  = lex.hdd(draws=42)        if n >= 50 else np.nan
    except Exception:
        f['mtld'], f['hdd'] = np.nan, np.nan

    counts = pd.Series(words).value_counts() if n else pd.Series(dtype=int)
    f['hapax_ratio']  = (counts == 1).sum() / n if n else 0
    f['top10_ratio']  = counts.head(10).sum() / n if n else 0
    f['long_word_ratio'] = np.mean([len(w) >= 7 for w in words]) if n else 0
    return f


feat_lex = pd.DataFrame([lexical_features(d) for d in docs])
print('shape:', feat_lex.shape)
print('missing values:\n', feat_lex.isnull().sum()[lambda s: s > 0])
print()
print(feat_lex.describe().round(3).to_string())

print('\n--- correlation with each target ---')
print(pd.DataFrame({c: feat_lex.corrwith(train[c]) for c in TARGETS}).round(3).to_string())

shape: (3911, 9)
missing values:
 mtld    2
hdd     2
dtype: int64

            ttr  lemma_ttr  root_ttr   log_ttr      mtld       hdd  hapax_ratio  top10_ratio  long_word_ratio
count  3911.000   3911.000  3911.000  3911.000  3909.000  3909.000     3911.000     3911.000         3911.000
mean      0.368      0.329     7.254     0.830    54.543     0.781        0.198        0.324            0.152
std       0.082      0.077     1.252     0.030    14.158     0.043        0.069        0.051            0.045
min       0.140      0.123     3.068     0.701    15.718     0.485        0.001        0.193            0.019
25%       0.311      0.275     6.410     0.811    44.589     0.757        0.150        0.289            0.121
50%       0.361      0.320     7.194     0.831    53.275     0.786        0.189        0.318            0.149
75%       0.419      0.376     8.062     0.850    62.752     0.811        0.237        0.352            0.179
max       0.929      0.929    12.415     0.972   123

## 27. Feature group 4 — syntactic complexity (POS + dependencies)

In [34]:
POS_TAGS = ['NOUN','VERB','ADJ','ADV','PRON','DET','ADP',
            'AUX','CCONJ','SCONJ','PART','NUM','PROPN','INTJ']

def syntactic_features(doc):
    toks = [t for t in doc if not t.is_punct and not t.is_space]
    n = len(toks)
    f = {}
    if n == 0:
        return {f'pos_{p.lower()}': 0 for p in POS_TAGS}

    # --- POS proportions ---
    pos_counts = pd.Series([t.pos_ for t in toks]).value_counts()
    for p in POS_TAGS:
        f[f'pos_{p.lower()}'] = pos_counts.get(p, 0) / n

    # --- dependency distance ---
    dists = [abs(t.i - t.head.i) for t in toks if t.head.i != t.i]
    f['dep_dist_mean'] = np.mean(dists) if dists else 0
    f['dep_dist_std']  = np.std(dists)  if dists else 0
    f['dep_dist_max']  = max(dists)     if dists else 0

    # --- structural variety ---
    deps = [t.dep_ for t in toks]
    f['n_unique_deps'] = len(set(deps))
    f['subord_ratio']  = sum(d in ('advcl','ccomp','xcomp','relcl','acl') for d in deps) / n
    f['coord_ratio']   = sum(d == 'conj' for d in deps) / n

    # --- clause structure ---
    sents = list(doc.sents)
    n_sent = max(len(sents), 1)
    f['verbs_per_sent'] = sum(t.pos_ in ('VERB','AUX') for t in toks) / n_sent
    f['clauses_per_sent'] = sum(d in ('advcl','ccomp','xcomp','relcl','acl','conj')
                                for d in deps) / n_sent

    # --- verb tense variety ---
    tenses = [t.morph.get('Tense') for t in toks if t.pos_ in ('VERB','AUX')]
    tenses = [x[0] for x in tenses if x]
    f['n_unique_tenses'] = len(set(tenses))
    f['past_ratio']    = tenses.count('Past') / len(tenses) if tenses else 0

    # --- sentence root depth ---
    depths = []
    for s in sents:
        for t in s:
            d, cur = 0, t
            while cur.head.i != cur.i and d < 50:
                cur, d = cur.head, d + 1
            depths.append(d)
    f['parse_depth_mean'] = np.mean(depths) if depths else 0
    f['parse_depth_max']  = max(depths) if depths else 0
    return f


feat_syn = pd.DataFrame([syntactic_features(d) for d in docs])
print('shape:', feat_syn.shape)
print()
print(feat_syn.describe().round(3).to_string())

print('\n--- correlation with each target ---')
corr_syn = pd.DataFrame({c: feat_syn.corrwith(train[c]) for c in TARGETS}).round(3)
print(corr_syn.to_string())

print('\n--- strongest features (by max |correlation|) ---')
print(corr_syn.abs().max(axis=1).sort_values(ascending=False).head(10).round(3).to_string())

shape: (3911, 26)

       pos_noun  pos_verb   pos_adj   pos_adv  pos_pron   pos_det   pos_adp   pos_aux  pos_cconj  pos_sconj  pos_part   pos_num  pos_propn  pos_intj  dep_dist_mean  dep_dist_std  dep_dist_max  n_unique_deps  subord_ratio  coord_ratio  verbs_per_sent  clauses_per_sent  n_unique_tenses  past_ratio  parse_depth_mean  parse_depth_max
count  3911.000  3911.000  3911.000  3911.000  3911.000  3911.000  3911.000  3911.000   3911.000   3911.000  3911.000  3911.000   3911.000  3911.000       3911.000      3911.000      3911.000       3911.000      3911.000     3911.000        3911.000          3911.000         3911.000    3911.000          3911.000         3911.000
mean      0.188     0.159     0.068     0.046     0.144     0.072     0.087     0.087      0.041      0.040     0.050     0.006      0.009     0.001          2.665         3.718        35.323         29.806         0.111        0.042           6.690             4.403            1.944       0.163             2.995   

In [35]:
punct_rate = feat_desc['n_sents'] / feat_desc['n_words']   # sentences per word

print('Is "complexity" just missing punctuation?')
print('-' * 52)
for c in ['dep_dist_mean', 'clauses_per_sent', 'subord_ratio', 'parse_depth_mean']:
    r = np.corrcoef(feat_syn[c], punct_rate)[0, 1]
    print(f'{c:<20} vs punctuation rate : {r:+.3f}')

# now look ONLY at essays that punctuate normally
ok = (feat_desc['words_per_sent'] > 8) & (feat_desc['words_per_sent'] < 30)
print(f'\nEssays with normal sentence length: {ok.sum()} of {len(ok)}')
print('\ncorrelation with syntax, normal essays only:')
for c in ['dep_dist_mean', 'clauses_per_sent', 'subord_ratio',
          'n_unique_deps', 'pos_sconj', 'pos_cconj']:
    r_all = np.corrcoef(feat_syn[c], train['syntax'])[0, 1]
    r_ok  = np.corrcoef(feat_syn.loc[ok, c], train.loc[ok, 'syntax'])[0, 1]
    print(f'  {c:<20} all: {r_all:+.3f}    normal-only: {r_ok:+.3f}')

Is "complexity" just missing punctuation?
----------------------------------------------------
dep_dist_mean        vs punctuation rate : -0.770
clauses_per_sent     vs punctuation rate : -0.687
subord_ratio         vs punctuation rate : -0.390
parse_depth_mean     vs punctuation rate : -0.818

Essays with normal sentence length: 2863 of 3911

correlation with syntax, normal essays only:
  dep_dist_mean        all: -0.321    normal-only: -0.177
  clauses_per_sent     all: -0.257    normal-only: -0.098
  subord_ratio         all: -0.208    normal-only: -0.130
  n_unique_deps        all: +0.149    normal-only: +0.179
  pos_sconj            all: -0.059    normal-only: -0.037
  pos_cconj            all: -0.085    normal-only: -0.035


In [36]:
def syntactic_features_v2(doc):
    toks = [t for t in doc if not t.is_punct and not t.is_space]
    n = len(toks)
    sents = list(doc.sents)
    f = {}
    if n == 0 or not sents:
        return {}

    # --- per-sentence, then averaged: immune to run-on inflation ---
    per_sent_dep, per_sent_depth, per_sent_len = [], [], []
    for s in sents:
        st = [t for t in s if not t.is_punct and not t.is_space]
        if len(st) < 2:
            continue
        L = len(st)
        per_sent_len.append(L)
        d = [abs(t.i - t.head.i) for t in st if t.head.i != t.i]
        per_sent_dep.append(np.mean(d) / L if d else 0)      # normalised by length
        depths = []
        for t in st:
            k, cur = 0, t
            while cur.head.i != cur.i and k < 50:
                cur, k = cur.head, k + 1
            depths.append(k)
        per_sent_depth.append(np.mean(depths) / np.log(L + 1))  # normalised

    f['norm_dep_dist']  = np.mean(per_sent_dep)   if per_sent_dep else 0
    f['norm_depth']     = np.mean(per_sent_depth) if per_sent_depth else 0

    # --- variety measures: robust to the confound ---
    deps = [t.dep_ for t in toks]
    f['n_unique_deps']    = len(set(deps))
    f['dep_entropy']      = -sum((c/n) * np.log(c/n)
                                 for c in pd.Series(deps).value_counts())
    f['n_unique_pos']     = len(set(t.pos_ for t in toks))
    f['pos_entropy']      = -sum((c/n) * np.log(c/n)
                                 for c in pd.Series([t.pos_ for t in toks]).value_counts())

    # --- ratios: scale-free ---
    f['subord_per_clause'] = (sum(d in ('advcl','ccomp','xcomp','relcl','acl') for d in deps)
                              / max(sum(d == 'conj' for d in deps)
                                    + sum(d in ('advcl','ccomp','xcomp','relcl','acl')
                                          for d in deps), 1))
    f['sconj_cconj_ratio'] = (sum(t.pos_ == 'SCONJ' for t in toks)
                              / max(sum(t.pos_ == 'CCONJ' for t in toks), 1))

    # --- the confound itself, as an explicit feature ---
    f['punct_rate']    = sum(1 for t in doc if t.is_punct) / n
    f['sents_per_100w'] = 100 * len(sents) / n
    f['is_runon']      = int(np.mean(per_sent_len) > 40) if per_sent_len else 0
    return f


feat_syn2 = pd.DataFrame([syntactic_features_v2(d) for d in docs]).fillna(0)
print('shape:', feat_syn2.shape)
print('\n--- correlation with each target ---')
c2 = pd.DataFrame({c: feat_syn2.corrwith(train[c]) for c in TARGETS}).round(3)
print(c2.to_string())
print('\nstrongest:')
print(c2.abs().max(axis=1).sort_values(ascending=False).round(3).to_string())

shape: (3911, 11)

--- correlation with each target ---
                   cohesion  syntax  vocabulary  phraseology  grammar  conventions
norm_dep_dist         0.033   0.071       0.022        0.016    0.063        0.082
norm_depth            0.002  -0.019       0.051        0.023    0.006       -0.007
n_unique_deps         0.170   0.149       0.231        0.201    0.094        0.085
dep_entropy           0.158   0.157       0.211        0.210    0.139        0.095
n_unique_pos         -0.068  -0.078      -0.030       -0.058   -0.097       -0.133
pos_entropy           0.048   0.041       0.080        0.092    0.049        0.003
subord_per_clause    -0.028  -0.039      -0.076       -0.060   -0.038       -0.024
sconj_cconj_ratio    -0.047  -0.036      -0.088       -0.064   -0.036        0.002
punct_rate            0.224   0.200       0.171        0.131    0.126        0.220
sents_per_100w        0.138   0.202       0.109        0.108    0.161        0.194
is_runon             -0.165  -0

## 28. Feature group 5 — referential cohesion

In [37]:
PERSONAL = {'i','you','he','she','it','we','they','me','him','her','us','them'}
POSSESS  = {'my','your','his','her','its','our','their','mine','yours','hers','ours','theirs'}
DEMONST  = {'this','that','these','those'}

def referential_features(doc):
    toks   = [t for t in doc if not t.is_punct and not t.is_space]
    n      = len(toks)
    sents  = list(doc.sents)
    n_sent = max(len(sents), 1)
    f = {}
    if n == 0:
        return f

    nouns = sum(t.pos_ in ('NOUN','PROPN') for t in toks)
    prons = sum(t.pos_ == 'PRON' for t in toks)
    lower = [t.text.lower() for t in toks]

    # --- article usage (the classic ELL difficulty) ---
    f['def_art_per_word']   = lower.count('the') / n
    f['indef_art_per_word'] = (lower.count('a') + lower.count('an')) / n
    f['def_art_per_sent']   = lower.count('the') / n_sent
    f['art_per_noun']       = (lower.count('the') + lower.count('a')
                               + lower.count('an')) / max(nouns, 1)

    # --- pronoun usage ---
    f['pron_per_word']      = prons / n
    f['pron_per_noun']      = prons / max(nouns, 1)
    f['personal_per_word']  = sum(w in PERSONAL for w in lower) / n
    f['possess_per_word']   = sum(w in POSSESS  for w in lower) / n
    f['demonst_per_word']   = sum(w in DEMONST  for w in lower) / n
    f['propn_per_noun']     = sum(t.pos_ == 'PROPN' for t in toks) / max(nouns, 1)

    # --- first vs third person ---
    fp = sum(w in {'i','me','my','mine','we','us','our','ours'} for w in lower)
    tp = sum(w in {'he','she','it','they','him','her','them',
                   'his','its','their','theirs'} for w in lower)
    f['first_person_ratio'] = fp / n
    f['third_person_ratio'] = tp / n

    # --- LEXICAL OVERLAP between adjacent sentences (Coh-Metrix style) ---
    sent_content, sent_nouns, sent_stems = [], [], []
    for s in sents:
        st = [t for t in s if not t.is_punct and not t.is_space]
        sent_content.append({t.lemma_.lower() for t in st if not t.is_stop})
        sent_nouns.append({t.lemma_.lower() for t in st if t.pos_ in ('NOUN','PROPN')})
        sent_stems.append({t.lemma_.lower()[:4] for t in st if not t.is_stop})

    def overlap(sets, k=1):
        vals = []
        for i in range(len(sets) - k):
            a, b = sets[i], sets[i + k]
            if a and b:
                vals.append(len(a & b) / len(a | b))     # Jaccard
        return vals

    o1 = overlap(sent_content, 1)
    o2 = overlap(sent_content, 2)
    on = overlap(sent_nouns, 1)
    os_ = overlap(sent_stems, 1)

    f['adj_content_overlap']     = np.mean(o1) if o1 else 0
    f['adj_content_overlap_std'] = np.std(o1)  if o1 else 0
    f['skip1_content_overlap']   = np.mean(o2) if o2 else 0
    f['adj_noun_overlap']        = np.mean(on) if on else 0
    f['adj_stem_overlap']        = np.mean(os_) if os_ else 0
    f['zero_overlap_ratio']      = np.mean([v == 0 for v in o1]) if o1 else 0
    return f


feat_ref = pd.DataFrame([referential_features(d) for d in docs]).fillna(0)
print('shape:', feat_ref.shape)
print()
print(feat_ref.describe().round(3).to_string())

print('\n--- correlation with each target ---')
cr = pd.DataFrame({c: feat_ref.corrwith(train[c]) for c in TARGETS}).round(3)
print(cr.to_string())

print('\nstrongest:')
print(cr.abs().max(axis=1).sort_values(ascending=False).round(3).to_string())

shape: (3911, 18)

       def_art_per_word  indef_art_per_word  def_art_per_sent  art_per_noun  pron_per_word  pron_per_noun  personal_per_word  possess_per_word  demonst_per_word  propn_per_noun  first_person_ratio  third_person_ratio  adj_content_overlap  adj_content_overlap_std  skip1_content_overlap  adj_noun_overlap  adj_stem_overlap  zero_overlap_ratio
count          3911.000            3911.000          3911.000      3911.000       3911.000       3911.000           3911.000          3911.000          3911.000        3911.000            3911.000            3911.000             3911.000                 3911.000               3911.000          3911.000          3911.000            3911.000
mean              0.033               0.023             0.945         0.294          0.144          0.827              0.088             0.023             0.024           0.043               0.033               0.046                0.113                    0.092                  0.097            

## 29. Feature groups 3, 6, 7, 8, 9 — combined

In [38]:
import textstat, re
from collections import Counter

CONNECTIVES = {
    'causal':      ['because','therefore','thus','since','so that','consequently',
                    'as a result','due to','hence','for this reason'],
    'contrastive': ['however','although','though','but','whereas','nevertheless',
                    'on the other hand','in contrast','yet','despite','even though'],
    'additive':    ['moreover','furthermore','also','in addition','besides',
                    'additionally','as well as','not only'],
    'temporal':    ['first','firstly','then','finally','next','meanwhile',
                    'afterwards','lastly','second','secondly','in conclusion'],
    'exemplify':   ['for example','for instance','such as','namely','in particular'],
}

CONTRACTION_ERRORS = ['dont','cant','wont','isnt','arent','didnt','doesnt','wouldnt',
                      'couldnt','shouldnt','hasnt','havent','wasnt','werent','im',
                      'ive','id','ill','youre','youve','theyre','thats','its a',
                      'lets','whats','hes','shes','were not']


def all_remaining_features(doc, text):
    f = {}
    toks  = [t for t in doc if not t.is_punct and not t.is_space]
    n     = max(len(toks), 1)
    sents = list(doc.sents)
    ns    = max(len(sents), 1)
    lower = text.lower()
    words = [t.text.lower() for t in toks]

    # ============ GROUP 3: READABILITY ============
    try:
        f['g3_flesch_ease']   = textstat.flesch_reading_ease(text)
        f['g3_flesch_grade']  = textstat.flesch_kincaid_grade(text)
        f['g3_gunning_fog']   = textstat.gunning_fog(text)
        f['g3_smog']          = textstat.smog_index(text)
        f['g3_coleman_liau']  = textstat.coleman_liau_index(text)
        f['g3_ari']           = textstat.automated_readability_index(text)
        f['g3_dale_chall']    = textstat.dale_chall_readability_score(text)
        f['g3_difficult_pct'] = textstat.difficult_words(text) / n
        f['g3_syll_per_word'] = textstat.syllable_count(text) / n
        f['g3_polysyll_pct']  = textstat.polysyllabcount(text) / n
    except Exception:
        for k in ['flesch_ease','flesch_grade','gunning_fog','smog','coleman_liau',
                  'ari','dale_chall','difficult_pct','syll_per_word','polysyll_pct']:
            f[f'g3_{k}'] = np.nan

    # ============ GROUP 6: SEMANTIC COHERENCE ============
    vecs = []
    for s in sents:
        st = [t for t in s if t.has_vector and not t.is_punct and not t.is_stop]
        if st:
            v = np.mean([t.vector for t in st], axis=0)
            nv = np.linalg.norm(v)
            vecs.append(v / nv if nv > 0 else v)
    if len(vecs) >= 2:
        V = np.array(vecs)
        sim1 = [float(V[i] @ V[i+1]) for i in range(len(V)-1)]
        sim2 = [float(V[i] @ V[i+2]) for i in range(len(V)-2)] or [0.0]
        cent = V.mean(axis=0); cent /= (np.linalg.norm(cent) + 1e-9)
        simc = [float(v @ cent) for v in V]
        f['g6_sim_adj_mean']  = np.mean(sim1)
        f['g6_sim_adj_std']   = np.std(sim1)
        f['g6_sim_adj_min']   = np.min(sim1)
        f['g6_sim_skip1']     = np.mean(sim2)
        f['g6_sim_centroid']  = np.mean(simc)
        f['g6_sim_cent_std']  = np.std(simc)
        f['g6_lowsim_ratio']  = float(np.mean([s < 0.5 for s in sim1]))
        f['g6_sim_drop']      = np.mean(sim1) - np.mean(sim2)
    else:
        for k in ['sim_adj_mean','sim_adj_std','sim_adj_min','sim_skip1',
                  'sim_centroid','sim_cent_std','lowsim_ratio','sim_drop']:
            f[f'g6_{k}'] = np.nan

    # ============ GROUP 7: CONNECTIVES ============
    total_conn = 0
    for cat, wl in CONNECTIVES.items():
        c = sum(lower.count(w) for w in wl)
        f[f'g7_{cat}_per_sent'] = c / ns
        total_conn += c
    f['g7_total_per_sent']   = total_conn / ns
    f['g7_total_per_word']   = total_conn / n
    f['g7_types_used']       = sum(
        1 for wl in CONNECTIVES.values() if any(w in lower for w in wl))
    f['g7_sent_start_conn']  = sum(
        1 for s in sents
        if s.text.strip().lower().split()[:1] and
        any(s.text.strip().lower().startswith(w)
            for wl in CONNECTIVES.values() for w in wl)) / ns

    # ============ GROUP 8: REPETITION / QUALITY ============
    wc = Counter(words)
    f['g8_top1_ratio']  = wc.most_common(1)[0][1] / n if wc else 0
    f['g8_top5_ratio']  = sum(c for _, c in wc.most_common(5)) / n
    f['g8_top20_ratio'] = sum(c for _, c in wc.most_common(20)) / n

    for k in (3, 5, 8):
        grams = [tuple(words[i:i+k]) for i in range(max(len(words)-k+1, 0))]
        if grams:
            gc = Counter(grams)
            f[f'g8_dup_{k}gram']      = 1 - len(gc) / len(grams)
            f[f'g8_top_{k}gram_char'] = (gc.most_common(1)[0][1] * k) / n
        else:
            f[f'g8_dup_{k}gram'] = 0
            f[f'g8_top_{k}gram_char'] = 0

    st_texts = [s.text.strip().lower() for s in sents if s.text.strip()]
    f['g8_dup_sent_ratio'] = (1 - len(set(st_texts)) / len(st_texts)) if st_texts else 0
    lines = [l.strip().lower() for l in text.split('\n') if l.strip()]
    f['g8_dup_line_ratio'] = (1 - len(set(lines)) / len(lines)) if lines else 0
    f['g8_max_word_repeat'] = wc.most_common(1)[0][1] if wc else 0

    # ============ GROUP 9: MECHANICS / ERRORS ============
    f['g9_contraction_err'] = sum(
        len(re.findall(r'\b' + w + r'\b', lower)) for w in CONTRACTION_ERRORS) / n
    f['g9_apostrophes']     = text.count("'") / n
    f['g9_lower_sent_start']= np.mean(
        [s.text.strip()[:1].islower() for s in sents if s.text.strip()]) if sents else 0
    f['g9_lone_i']          = len(re.findall(r'\bi\b', text)) / n
    f['g9_no_space_punct']  = len(re.findall(r'[,.!?][A-Za-z]', text)) / n
    f['g9_double_space']    = text.count('  ') / n
    f['g9_repeat_punct']    = len(re.findall(r'[!?.]{2,}', text)) / n
    f['g9_comma_per_sent']  = text.count(',') / ns
    f['g9_upper_ratio']     = sum(c.isupper() for c in text) / max(len(text), 1)
    f['g9_oov_ratio']       = np.mean([t.is_oov for t in toks]) if toks else 0
    f['g9_nonalpha_words']  = np.mean([not t.text.isalpha() for t in toks]) if toks else 0
    f['g9_all_caps_words']  = np.mean([t.text.isupper() and len(t.text) > 1
                                       for t in toks]) if toks else 0
    return f


feat_rest = pd.DataFrame([all_remaining_features(d, t) for d, t in zip(docs, texts)])
print('shape:', feat_rest.shape)
print('missing:', feat_rest.isnull().sum().sum())

corr_rest = pd.DataFrame({c: feat_rest.corrwith(train[c]) for c in TARGETS}).round(3)

for g, name in [('g3','GROUP 3 — READABILITY'),
                ('g6','GROUP 6 — SEMANTIC COHERENCE'),
                ('g7','GROUP 7 — CONNECTIVES'),
                ('g8','GROUP 8 — REPETITION'),
                ('g9','GROUP 9 — MECHANICS')]:
    sub = corr_rest[corr_rest.index.str.startswith(g)]
    print('\n' + '=' * 78)
    print(name)
    print('=' * 78)
    print(sub.to_string())
    print(f'  --> strongest: {sub.abs().max(axis=1).idxmax()} '
          f'({sub.abs().max(axis=1).max():.3f})')

shape: (3911, 51)
missing: 80

GROUP 3 — READABILITY
                  cohesion  syntax  vocabulary  phraseology  grammar  conventions
g3_flesch_ease       0.137   0.203       0.093        0.113    0.144        0.178
g3_flesch_grade     -0.180  -0.243      -0.141       -0.144   -0.171       -0.221
g3_gunning_fog      -0.181  -0.244      -0.141       -0.146   -0.172       -0.224
g3_smog             -0.072  -0.157      -0.019       -0.071   -0.122       -0.127
g3_coleman_liau      0.178   0.150       0.195        0.131    0.105        0.160
g3_ari              -0.176  -0.238      -0.141       -0.141   -0.166       -0.217
g3_dale_chall       -0.098  -0.152      -0.053       -0.083   -0.097       -0.164
g3_difficult_pct     0.171   0.172       0.231        0.176    0.159        0.123
g3_syll_per_word     0.215   0.197       0.245        0.156    0.130        0.217
g3_polysyll_pct      0.169   0.145       0.201        0.115    0.091        0.156
  --> strongest: g3_syll_per_word (0.245)

GR

/usr/local/lib/python3.12/dist-packages/numpy/lib/_function_base_impl.py:2922: RuntimeWarning: invalid value encountered in divide
/usr/local/lib/python3.12/dist-packages/numpy/lib/_function_base_impl.py:2923: RuntimeWarning: invalid value encountered in divide


## 30. Assemble feature matrix

In [39]:
X_feat = pd.concat([feat_desc, feat_lex, feat_syn, feat_ref, feat_rest], axis=1)

# drop the dead column and the skipped group
X_feat = X_feat.drop(columns=['g9_oov_ratio'], errors='ignore')
X_feat = X_feat.drop(columns=[c for c in X_feat.columns if c.startswith('g6_')])

# guard against duplicate column names from overlapping groups
dups = X_feat.columns[X_feat.columns.duplicated()].tolist()
if dups:
    print('duplicate columns found, keeping first:', dups)
    X_feat = X_feat.loc[:, ~X_feat.columns.duplicated()]

X_feat = X_feat.replace([np.inf, -np.inf], np.nan)

print('final feature matrix:', X_feat.shape)
print('missing values      :', X_feat.isnull().sum().sum())
print('constant columns    :', (X_feat.nunique() <= 1).sum())
print()
print('feature groups:')
for g, name in [('n_|chars_|words_|sent_|paras','descriptive'),
                ('ttr|mtld|hdd|hapax|top10|long_word','lexical'),
                ('pos_|dep_|subord|coord|clause|verbs_|tense|past|parse','syntactic'),
                ('art_|pron_|person|propn_|demonst|possess|overlap','referential'),
                ('g3_','readability'), ('g7_','connectives'),
                ('g8_','repetition'), ('g9_','mechanics')]:
    print(f'  {name:<14} {X_feat.columns.str.contains(g).sum():>4}')

X_feat.to_csv('features_115.csv', index=False)
print('\nsaved -> features_115.csv')

final feature matrix: (3911, 106)
missing values      : 4
constant columns    : 0

feature groups:
  descriptive      23
  lexical           9
  syntactic        25
  referential      19
  readability      10
  connectives       9
  repetition       12
  mechanics        11

saved -> features_115.csv


# LGBM v/s XG v/s Ridge

In [40]:
import lightgbm as lgb
import xgboost as xgb
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

def run_feat_model(make_model, tag, verbose=True):
    oof = np.zeros((len(train), len(TARGETS)))
    for f in range(N_FOLDS):
        tr = train.index[train['fold'] != f]
        va = train.index[train['fold'] == f]
        for j, t in enumerate(TARGETS):
            m = make_model()
            m.fit(X_feat.loc[tr], train.loc[tr, t])
            oof[va, j] = m.predict(X_feat.loc[va])
    oof = np.clip(oof, 1.0, 5.0)
    s, cols = mcrmse(train[TARGETS].values, oof, per_column=True)
    if verbose:
        print(f'{tag:<28} {s:.4f}')
    return s, oof, cols

print(f'{"model":<28} MCRMSE')
print('=' * 40)
print(f'{"baseline (mean)":<28} 0.6528')
print(f'{"TF-IDF + Ridge":<28} 0.5288')
print('-' * 40)

s_r, oof_r, _ = run_feat_model(
    lambda: make_pipeline(SimpleImputer(strategy='median'),
                          StandardScaler(), Ridge(alpha=10.0)),
    'features + Ridge')

s_l, oof_l, cols_l = run_feat_model(
    lambda: lgb.LGBMRegressor(n_estimators=400, learning_rate=0.05,
                              num_leaves=15, min_child_samples=30,
                              colsample_bytree=0.7, subsample=0.8, subsample_freq=1,
                              reg_lambda=1.0, verbose=-1,
                              random_state=SEED, n_jobs=-1),
    'features + LightGBM')

s_x, oof_x, cols_x = run_feat_model(
    lambda: xgb.XGBRegressor(n_estimators=400, learning_rate=0.05,
                             max_depth=4, min_child_weight=10,
                             colsample_bytree=0.7, subsample=0.8,
                             reg_lambda=1.0, verbosity=0,
                             random_state=SEED, n_jobs=-1),
    'features + XGBoost')

# simple average of the two boosters
s_avg = mcrmse(train[TARGETS].values, np.clip((oof_l + oof_x) / 2, 1, 5))
print(f'{"LGBM + XGB averaged":<28} {s_avg:.4f}')

best_oof = {'lgb': oof_l, 'xgb': oof_x}[min([('lgb', s_l), ('xgb', s_x)],
                                            key=lambda z: z[1])[0]]
print('\nper-column (LightGBM):')
for n, v in zip(TARGETS, cols_l):
    print(f'  {n:<14} {v:.4f}')

np.save('oof_feat_lgb.npy',   oof_l)
np.save('oof_feat_xgb.npy',   oof_x)
np.save('oof_feat_ridge.npy', oof_r)
print('\nsaved 3 OOF files')

model                        MCRMSE
baseline (mean)              0.6528
TF-IDF + Ridge               0.5288
----------------------------------------
features + Ridge             0.5326
features + LightGBM          0.5330
features + XGBoost           0.5329
LGBM + XGB averaged          0.5300

per-column (LightGBM):
  cohesion       0.5385
  syntax         0.5149
  vocabulary     0.4726
  phraseology    0.5352
  grammar        0.5977
  conventions    0.5389

saved 3 OOF files


# Feature Importance + blend test

In [41]:
m = lgb.LGBMRegressor(n_estimators=400, learning_rate=0.05, num_leaves=15,
                      min_child_samples=30, colsample_bytree=0.7, subsample=0.8,
                      subsample_freq=1, reg_lambda=1.0, verbose=-1,
                      random_state=SEED, n_jobs=-1)
imp = np.zeros(X_feat.shape[1])
for t in TARGETS:
    m.fit(X_feat, train[t])
    imp += m.feature_importances_
imp = pd.Series(imp, index=X_feat.columns).sort_values(ascending=False)

print('TOP 20 FEATURES')
print('-' * 34)
print(imp.head(20).round(0).to_string())
print('\nBOTTOM 10')
print(imp.tail(10).round(0).to_string())

# --- does combining TF-IDF with features help? ---
oof_tfidf = np.load('oof_tfidf_ridge.npy')
oof_feat  = (oof_l + oof_x) / 2

print('\n\nBLEND TEST')
print('-' * 34)
print(f'{"TF-IDF alone":<22} {mcrmse(train[TARGETS].values, oof_tfidf):.4f}')
print(f'{"features alone":<22} {mcrmse(train[TARGETS].values, oof_feat):.4f}')
for w in [0.1, 0.2, 0.3, 0.4, 0.5]:
    b = np.clip((1-w)*oof_tfidf + w*oof_feat, 1, 5)
    print(f'{"tfidf " + str(round(1-w,1)) + " + feat " + str(w):<22} '
          f'{mcrmse(train[TARGETS].values, b):.4f}')

TOP 20 FEATURES
----------------------------------
pos_aux                607.0
root_ttr               604.0
subord_ratio           576.0
g9_lower_sent_start    561.0
third_person_ratio     550.0
g9_comma_per_sent      537.0
g8_dup_3gram           533.0
pos_verb               526.0
art_per_noun           520.0
demonst_per_word       497.0
pos_adj                489.0
pos_part               482.0
mtld                   479.0
indef_art_per_word     479.0
first_person_ratio     475.0
g7_sent_start_conn     474.0
pos_adp                467.0
pos_cconj              459.0
pos_sconj              451.0
pos_adv                444.0

BOTTOM 10
g3_flesch_grade      157.0
g9_lone_i            151.0
parse_depth_max      117.0
g9_all_caps_words    100.0
pron_per_word         84.0
g7_types_used         77.0
g9_repeat_punct       47.0
g8_dup_sent_ratio     20.0
n_unique_tenses        0.0
g8_dup_line_ratio      0.0


BLEND TEST
----------------------------------
TF-IDF alone           0.5288
features a

# Trying multiple blends

In [42]:
oof_tfidf = np.load('oof_tfidf_ridge.npy')
Y = train[TARGETS].values

def gmean(arrs, ws):
    ws = np.array(ws) / np.sum(ws)
    out = np.ones_like(arrs[0])
    for a, w in zip(arrs, ws):
        out *= np.clip(a, 1e-6, None) ** w
    return out

# ---------- STEP 1: best feature-side model ----------
print('FEATURE-SIDE BLEND (ridge / lgbm / xgb)')
print('-' * 44)
for nm, a in [('ridge', oof_r), ('lgbm', oof_l), ('xgb', oof_x)]:
    print(f'{nm:<28} {mcrmse(Y, a):.4f}')

best_f, best_s = None, 9
for wr in np.arange(0, 1.01, 0.1):
    for wl in np.arange(0, 1.01 - wr + 1e-9, 0.1):
        wx = 1 - wr - wl
        b = np.clip(wr*oof_r + wl*oof_l + wx*oof_x, 1, 5)
        s = mcrmse(Y, b)
        if s < best_s:
            best_s, best_f, best_w = s, b, (wr, wl, wx)
print(f'{"best weighted (A)":<28} {best_s:.4f}   '
      f'ridge={best_w[0]:.1f} lgbm={best_w[1]:.1f} xgb={best_w[2]:.1f}')

g = np.clip(gmean([oof_r, oof_l, oof_x], best_w), 1, 5)
print(f'{"same weights, geometric":<28} {mcrmse(Y, g):.4f}')

# ---------- STEP 2: blend with TF-IDF, full range ----------
print('\n\nTF-IDF  +  FEATURE-BLEND')
print('-' * 44)
print(f'{"w_feat":>8}{"arithmetic":>14}{"geometric":>13}')
for w in np.arange(0, 1.01, 0.1):
    a = np.clip((1-w)*oof_tfidf + w*best_f, 1, 5)
    gg = np.clip(gmean([oof_tfidf, best_f], [1-w, w]), 1, 5)
    print(f'{w:>8.1f}{mcrmse(Y, a):>14.4f}{mcrmse(Y, gg):>13.4f}')

# ---------- STEP 3: all 4 models, free weights ----------
print('\n\nALL 4 MODELS, GRID SEARCH')
print('-' * 44)
mods = [oof_tfidf, oof_r, oof_l, oof_x]
names = ['tfidf', 'ridge', 'lgbm', 'xgb']
best_s4, best_w4 = 9, None
for w0 in np.arange(0, 1.01, 0.1):
    for w1 in np.arange(0, 1.01-w0+1e-9, 0.1):
        for w2 in np.arange(0, 1.01-w0-w1+1e-9, 0.1):
            w3 = 1 - w0 - w1 - w2
            b = np.clip(sum(w*m for w, m in zip([w0,w1,w2,w3], mods)), 1, 5)
            s = mcrmse(Y, b)
            if s < best_s4:
                best_s4, best_w4 = s, (w0, w1, w2, w3)
print('best weights:', {n: round(w,2) for n, w in zip(names, best_w4)})
print(f'best MCRMSE : {best_s4:.4f}')

final = np.clip(sum(w*m for w, m in zip(best_w4, mods)), 1, 5)
_, c = mcrmse(Y, final, per_column=True)
print('\nper-column:')
for n, v in zip(TARGETS, c):
    print(f'  {n:<14} {v:.4f}')
np.save('oof_blend_stage1.npy', final)

FEATURE-SIDE BLEND (ridge / lgbm / xgb)
--------------------------------------------
ridge                        0.5326
lgbm                         0.5330
xgb                          0.5329
best weighted (A)            0.5214   ridge=0.5 lgbm=0.3 xgb=0.2
same weights, geometric      0.5214


TF-IDF  +  FEATURE-BLEND
--------------------------------------------
  w_feat    arithmetic    geometric
     0.0        0.5288       0.5288
     0.1        0.5226       0.5224
     0.2        0.5176       0.5173
     0.3        0.5137       0.5134
     0.4        0.5111       0.5108
     0.5        0.5097       0.5094
     0.6        0.5096       0.5094
     0.7        0.5107       0.5105
     0.8        0.5131       0.5130
     0.9        0.5166       0.5166
     1.0        0.5214       0.5214


ALL 4 MODELS, GRID SEARCH
--------------------------------------------
best weights: {'tfidf': np.float64(0.4), 'ridge': np.float64(0.2), 'lgbm': np.float64(0.3), 'xgb': np.float64(0.1)}
best MCRMSE :

## Testing different aggregation methods for the 4 Models 

In [43]:
mods  = [oof_tfidf, oof_r, oof_l, oof_x]
names = ['tfidf', 'ridge', 'lgbm', 'xgb']
Y = train[TARGETS].values

def blend(ws, mode):
    ws = np.array(ws)
    if mode == 'A':
        return np.clip(sum(w*m for w, m in zip(ws, mods)), 1, 5)
    if mode == 'G':
        out = np.ones_like(mods[0])
        for m, w in zip(mods, ws):
            out *= np.clip(m, 1e-6, None) ** w
        return np.clip(out, 1, 5)
    if mode == 'H':                      # harmonic
        return np.clip(1.0 / sum(w / np.clip(m, 1e-6, None)
                                 for w, m in zip(ws, mods)), 1, 5)
    if mode == 'P2':                     # power mean, p=2 (quadratic)
        return np.clip(np.sqrt(sum(w * m**2 for w, m in zip(ws, mods))), 1, 5)

print(f'{"mean type":<24}{"MCRMSE":>10}{"best weights":>34}')
print('-' * 68)
results = {}
for mode, label in [('A', 'arithmetic'), ('G', 'geometric'),
                    ('H', 'harmonic'), ('P2', 'quadratic (p=2)')]:
    best_s, best_w = 9, None
    for w0 in np.arange(0, 1.01, 0.1):
        for w1 in np.arange(0, 1.01-w0+1e-9, 0.1):
            for w2 in np.arange(0, 1.01-w0-w1+1e-9, 0.1):
                w3 = 1 - w0 - w1 - w2
                s = mcrmse(Y, blend([w0, w1, w2, w3], mode))
                if s < best_s:
                    best_s, best_w = s, (w0, w1, w2, w3)
    results[label] = (best_s, best_w)
    ws = '  '.join(f'{n}={w:.1f}' for n, w in zip(names, best_w))
    print(f'{label:<24}{best_s:>10.4f}{ws:>34}')

# how different are the predictions themselves?
best_A = blend(results['arithmetic'][1], 'A')
best_G = blend(results['geometric'][1],  'G')
print(f'\nmax |A - G| across all predictions: {np.abs(best_A-best_G).max():.4f}')
print(f'mean |A - G|                      : {np.abs(best_A-best_G).mean():.4f}')

mean type                   MCRMSE                      best weights
--------------------------------------------------------------------
arithmetic                  0.5095tfidf=0.4  ridge=0.2  lgbm=0.3  xgb=0.1
geometric                   0.5092tfidf=0.5  ridge=0.2  lgbm=0.2  xgb=0.1
harmonic                    0.5089tfidf=0.5  ridge=0.2  lgbm=0.2  xgb=0.1
quadratic (p=2)             0.5099tfidf=0.4  ridge=0.2  lgbm=0.3  xgb=0.1

max |A - G| across all predictions: 0.1763
mean |A - G|                      : 0.0224


In [44]:
import warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model import RidgeCV

Y = train[TARGETS].values
oof_list  = [oof_tfidf, oof_r, oof_l, oof_x]
oof_names = ['tfidf', 'ridge', 'lgbm', 'xgb']

# 3911 x 24 meta-feature matrix
META = np.hstack(oof_list)
meta_cols = [f'{n}_{t}' for n in oof_names for t in TARGETS]
print('meta matrix:', META.shape)

def run_stack(make_model, tag, use_all=True):
    """use_all=True  -> each target sees all 24 columns
       use_all=False -> each target sees only its own 4 columns"""
    oof = np.zeros_like(Y)
    for f in range(N_FOLDS):
        tr = train.index[train['fold'] != f]
        va = train.index[train['fold'] == f]
        for j, t in enumerate(TARGETS):
            cols = np.arange(META.shape[1]) if use_all else \
                   np.array([k*6 + j for k in range(len(oof_list))])
            m = make_model()
            m.fit(META[tr][:, cols], Y[tr, j])
            oof[va, j] = m.predict(META[va][:, cols])
    oof = np.clip(oof, 1, 5)
    s = mcrmse(Y, oof)
    print(f'{tag:<42} {s:.4f}')
    return s, oof

print(f'\n{"method":<42} MCRMSE')
print('=' * 54)
print(f'{"weighted average (grid search)":<42} 0.5095')
print('-' * 54)

s1, oof_s1 = run_stack(lambda: Ridge(alpha=1.0),
                       'stack: Ridge, own target only (4 cols)', use_all=False)
s2, oof_s2 = run_stack(lambda: Ridge(alpha=1.0),
                       'stack: Ridge, all 24 cols', use_all=True)
s3, oof_s3 = run_stack(lambda: Ridge(alpha=20.0),
                       'stack: Ridge alpha=20, all 24 cols', use_all=True)
s4, oof_s4 = run_stack(lambda: RidgeCV(alphas=np.logspace(-2, 3, 20)),
                       'stack: RidgeCV, all 24 cols', use_all=True)
s5, oof_s5 = run_stack(lambda: lgb.LGBMRegressor(
                           n_estimators=200, learning_rate=0.03, num_leaves=7,
                           min_child_samples=50, colsample_bytree=0.8,
                           reg_lambda=5.0, verbose=-1, random_state=SEED),
                       'stack: LightGBM, all 24 cols', use_all=True)

# what weights did the linear stacker learn?
print('\nlearned coefficients (Ridge, all 24 cols, fitted on full data):')
m = Ridge(alpha=1.0).fit(META, Y)
coef = pd.DataFrame(m.coef_, index=TARGETS, columns=meta_cols)
print(coef.round(2).to_string())

best = min([(s1, oof_s1), (s2, oof_s2), (s3, oof_s3), (s4, oof_s4), (s5, oof_s5)],
           key=lambda z: z[0])
np.save('oof_stack.npy', best[1])
print(f'\nbest stack: {best[0]:.4f}   (weighted average was 0.5095)')

meta matrix: (3911, 24)

method                                     MCRMSE
weighted average (grid search)             0.5095
------------------------------------------------------
stack: Ridge, own target only (4 cols)     0.5055
stack: Ridge, all 24 cols                  0.5050
stack: Ridge alpha=20, all 24 cols         0.5042
stack: RidgeCV, all 24 cols                0.5042
stack: LightGBM, all 24 cols               0.5072

learned coefficients (Ridge, all 24 cols, fitted on full data):
             tfidf_cohesion  tfidf_syntax  tfidf_vocabulary  tfidf_phraseology  tfidf_grammar  tfidf_conventions  ridge_cohesion  ridge_syntax  ridge_vocabulary  ridge_phraseology  ridge_grammar  ridge_conventions  lgbm_cohesion  lgbm_syntax  lgbm_vocabulary  lgbm_phraseology  lgbm_grammar  lgbm_conventions  xgb_cohesion  xgb_syntax  xgb_vocabulary  xgb_phraseology  xgb_grammar  xgb_conventions
cohesion               0.23          0.03              0.09               0.01           0.06              

## 1. Setup

In [45]:
!pip install -q iterative-stratification sentencepiece

import os, gc, sys, math, random, time, warnings
warnings.filterwarnings('ignore')
import numpy as np, pandas as pd
import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.amp import autocast, GradScaler
from transformers import AutoTokenizer, AutoModel, AutoConfig, get_cosine_schedule_with_warmup

def seed_everything(seed=42):
    random.seed(seed); np.random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(42)
device = torch.device('cuda')
TARGETS = ['cohesion','syntax','vocabulary','phraseology','grammar','conventions']

def mcrmse(y_true, y_pred, per_column=False):
    c = np.sqrt(np.mean((y_true - y_pred) ** 2, axis=0))
    return (c.mean(), c) if per_column else c.mean()

print('GPU :', torch.cuda.get_device_name(0))
print('free:', f'{torch.cuda.mem_get_info()[0]/1e9:.2f} GB')

GPU : Tesla T4
free: 15.53 GB


In [46]:
import gc, torch
gc.collect(); torch.cuda.empty_cache()
print('GPU free:', f'{torch.cuda.mem_get_info()[0]/1e9:.2f} GB')

GPU free: 15.53 GB


## Config

In [47]:
!pip install -q iterative-stratification
from iterstrat.ml_stratifiers import MultilabelStratifiedKFold

DATA_DIR = '/kaggle/input/competitions/feedback-prize-english-language-learning'
train = pd.read_csv(f'{DATA_DIR}/train.csv')
y_ind = pd.get_dummies(train[TARGETS].astype(str)).values.astype(int)
mskf = MultilabelStratifiedKFold(n_splits=5, shuffle=True, random_state=42)
train['fold'] = -1
for f, (_, v) in enumerate(mskf.split(train, y_ind)):
    train.loc[v, 'fold'] = f

TARGET_MEANS = train[TARGETS].mean().values.astype('float32')

os.makedirs('/kaggle/working/saved', exist_ok=True)
train.to_csv('/kaggle/working/saved/train_5folds.csv', index=False)

print(train['fold'].value_counts().sort_index().to_string())
print('\nfold 0 cohesion:', round(train[train.fold==0]['cohesion'].mean(), 4),
      ' (must be 3.1216)')

fold
0    781
1    787
2    785
3    780
4    778

fold 0 cohesion: 3.1216  (must be 3.1216)


In [48]:
class CFG:
    model_name    = 'microsoft/deberta-v3-base'
    max_len       = 512
    batch_size    = 8
    valid_bs      = 16
    epochs        = 3          # was 4
    lr            = 2e-5
    head_lr       = 2e-5
    weight_decay  = 0.01
    warmup_ratio  = 0.0
    grad_accum    = 1
    max_grad_norm = 1000.0
    eval_every    = 40
    n_folds       = 5
    seed          = 42

tokenizer = AutoTokenizer.from_pretrained(CFG.model_name)
print('epochs:', CFG.epochs, '| steps:', (3130 // CFG.batch_size) * CFG.epochs)
assert CFG.grad_accum == 1 and CFG.head_lr == 2e-5
print('config OK')

config.json:   0%|          | 0.00/579 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

epochs: 3 | steps: 1173
config OK


## Dataset + dynamic padding

In [49]:
class EssayDataset(Dataset):
    def __init__(self, df, tokenizer, max_len, has_labels=True):
        self.texts = df['full_text'].values
        self.tok = tokenizer
        self.max_len = max_len
        self.has_labels = has_labels
        self.labels = df[TARGETS].values.astype('float32') if has_labels else None

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, i):
        enc = self.tok(self.texts[i], add_special_tokens=True,
                       max_length=self.max_len, truncation=True, padding=False)
        item = {'input_ids': enc['input_ids'],
                'attention_mask': enc['attention_mask']}
        if self.has_labels:
            item['labels'] = self.labels[i]
        return item


def collate(batch):
    maxlen = max(len(b['input_ids']) for b in batch)
    pad_id = tokenizer.pad_token_id
    ids  = torch.tensor([b['input_ids'] + [pad_id]*(maxlen-len(b['input_ids']))
                         for b in batch], dtype=torch.long)
    mask = torch.tensor([b['attention_mask'] + [0]*(maxlen-len(b['attention_mask']))
                         for b in batch], dtype=torch.long)
    out = {'input_ids': ids, 'attention_mask': mask}
    if 'labels' in batch[0]:
        out['labels'] = torch.tensor(np.array([b['labels'] for b in batch]),
                                     dtype=torch.float)
    return out

print('dataset + collate ready')

dataset + collate ready


## Model

In [50]:
class EssayModel(nn.Module):
    def __init__(self, model_name, n_targets=6):
        super().__init__()
        self.config = AutoConfig.from_pretrained(model_name)
        self.backbone = AutoModel.from_pretrained(
            model_name, config=self.config, dtype=torch.float32)
        self.backbone.gradient_checkpointing_enable(
            gradient_checkpointing_kwargs={'use_reentrant': False})
        self.head = nn.Linear(self.config.hidden_size, n_targets)
        nn.init.normal_(self.head.weight, std=0.02)
        with torch.no_grad():
            self.head.bias.copy_(torch.tensor(TARGET_MEANS))

    def mean_pool(self, tv, mask):
        mask = mask.unsqueeze(-1).expand(tv.size()).float()
        return torch.sum(tv * mask, 1) / torch.clamp(mask.sum(1), min=1e-9)

    def forward(self, input_ids, attention_mask):
        out = self.backbone(input_ids=input_ids, attention_mask=attention_mask)
        return self.head(self.mean_pool(out.last_hidden_state, attention_mask))

print('model class ready | GPU free:',
      f'{torch.cuda.mem_get_info()[0]/1e9:.2f} GB')

model class ready | GPU free: 15.53 GB


## Training function

In [51]:
def get_optimizer(model, cfg):
    nd = ['bias', 'LayerNorm.weight']
    return torch.optim.AdamW([
        {'params': [p for n,p in model.backbone.named_parameters()
                    if not any(x in n for x in nd)],
         'lr': cfg.lr, 'weight_decay': cfg.weight_decay},
        {'params': [p for n,p in model.backbone.named_parameters()
                    if any(x in n for x in nd)],
         'lr': cfg.lr, 'weight_decay': 0.0},
        {'params': model.head.parameters(),
         'lr': cfg.head_lr, 'weight_decay': 0.0},
    ], eps=1e-6, betas=(0.9, 0.999))


@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    preds = []
    for b in loader:
        with autocast('cuda', dtype=torch.float16):
            out = model(b['input_ids'].to(device), b['attention_mask'].to(device))
        preds.append(out.float().cpu().numpy())
    model.train()
    return np.concatenate(preds)


def train_fold(fold, cfg, verbose=True):
    sys.last_traceback = None; sys.last_value = None; sys.last_type = None
    seed_everything(cfg.seed + fold)
    gc.collect(); torch.cuda.empty_cache()

    va_idx = train.index[train.fold == fold].values        # <-- THE FIX
    tr_df  = train[train.fold != fold].reset_index(drop=True)
    va_df  = train[train.fold == fold].reset_index(drop=True)
    assert len(va_idx) == len(va_df)

    tr_dl = DataLoader(EssayDataset(tr_df, tokenizer, cfg.max_len),
                       batch_size=cfg.batch_size, shuffle=True,
                       num_workers=2, pin_memory=True, drop_last=True,
                       collate_fn=collate)
    va_dl = DataLoader(EssayDataset(va_df, tokenizer, cfg.max_len),
                       batch_size=cfg.valid_bs, shuffle=False,
                       num_workers=2, pin_memory=True, collate_fn=collate)

    model = EssayModel(cfg.model_name).to(device)
    optimizer = get_optimizer(model, cfg)
    criterion = nn.MSELoss()
    scaler = GradScaler('cuda')

    total_steps = (len(tr_dl) // cfg.grad_accum) * cfg.epochs
    scheduler = get_cosine_schedule_with_warmup(
        optimizer, int(total_steps * cfg.warmup_ratio), total_steps)

    y_va = va_df[TARGETS].values
    best_score, best_preds, step = 999.0, None, 0
    t0 = time.time()
    if verbose:
        print(f'fold {fold} | {len(tr_df)} train, {len(va_df)} val | {total_steps} steps')
        print(f'  val rows {va_idx[:3]} ... {va_idx[-3:]}')

    try:
        for epoch in range(cfg.epochs):
            model.train(); optimizer.zero_grad()
            for i, b in enumerate(tr_dl):
                with autocast('cuda', dtype=torch.float16):
                    out = model(b['input_ids'].to(device, non_blocking=True),
                                b['attention_mask'].to(device, non_blocking=True))
                    loss = criterion(out, b['labels'].to(device)) / cfg.grad_accum
                scaler.scale(loss).backward()

                if (i + 1) % cfg.grad_accum == 0:
                    scaler.unscale_(optimizer)
                    torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.max_grad_norm)
                    scaler.step(optimizer); scaler.update()
                    optimizer.zero_grad(); scheduler.step()
                    step += 1
                    if step % cfg.eval_every == 0:
                        p = evaluate(model, va_dl)
                        s = mcrmse(y_va, np.clip(p, 1, 5))
                        tag = ''
                        if s < best_score:
                            best_score, best_preds, tag = s, p, '  *'
                        if verbose:
                            print(f'  ep{epoch} step{step:>4}  val {s:.4f}  '
                                  f'best {best_score:.4f}  {time.time()-t0:.0f}s{tag}')

            p = evaluate(model, va_dl)
            s = mcrmse(y_va, np.clip(p, 1, 5))
            tag = ''
            if s < best_score:
                best_score, best_preds, tag = s, p, '  *'
            if verbose:
                print(f'  ep{epoch} END      val {s:.4f}  best {best_score:.4f}  '
                      f'{time.time()-t0:.0f}s{tag}')
    finally:
        del model, optimizer, scaler
        gc.collect(); torch.cuda.empty_cache()

    return best_score, best_preds, va_idx

print('train_fold FIXED')

train_fold FIXED


## Train all 5 folds

In [52]:
os.makedirs('/kaggle/working/saved', exist_ok=True)
oof_deberta = np.full((len(train), 6), np.nan)
scores, t0 = [], time.time()

for f in range(5):
    print(f'\n===== FOLD {f} =====')
    s, p, idx = train_fold(f, CFG)

    assert len(idx) == len(p), 'index/prediction length mismatch'
    assert np.isnan(oof_deberta[idx]).all(), f'fold {f} overwriting existing rows!'
    oof_deberta[idx] = p
    scores.append(s)

    np.save(f'/kaggle/working/saved/deberta_fold{f}_preds.npy', p)
    np.save(f'/kaggle/working/saved/deberta_fold{f}_idx.npy', idx)
    np.save('/kaggle/working/saved/oof_deberta.npy', oof_deberta)

    filled = int((~np.isnan(oof_deberta[:, 0])).sum())
    done = (time.time() - t0) / 60
    print(f'>>> fold {f}: {s:.4f} | rows filled {filled}/{len(train)} | '
          f'{done:.0f} min | ~{done/(f+1)*(5-f-1):.0f} min left')

assert not np.isnan(oof_deberta).any(), 'some rows never filled'
oof_deberta = np.clip(oof_deberta, 1, 5)
cv, cols = mcrmse(train[TARGETS].values, oof_deberta, per_column=True)

print(f'\nfold scores : {[round(x,4) for x in scores]}')
print(f'mean of folds: {np.mean(scores):.4f}')
print(f'OOF CV       : {cv:.4f}   <-- must be close to mean of folds')
for n, v in zip(TARGETS, cols):
    print(f'  {n:<14} {v:.4f}')
np.save('/kaggle/working/saved/oof_deberta.npy', oof_deberta)


===== FOLD 0 =====


pytorch_model.bin:   0%|          | 0.00/371M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

DebertaV2Model LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
mask_predictions.dense.bias             | UNEXPECTED |  | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED |  | 
mask_predictions.LayerNorm.weight       | UNEXPECTED |  | 
mask_predictions.dense.weight           | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED |  | 
lm_predictions.lm_head.bias             | UNEXPECTED |  | 
mask_predictions.classifier.bias        | UNEXPECTED |  | 
mask_predictions.LayerNorm.bias         | UNEXPECTED |  | 
mask_predictions.classifier.weight      | UNEXPECTED |  | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


model.safetensors:   0%|          | 0.00/371M [00:00<?, ?B/s]

...

DebertaV2Model LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
mask_predictions.dense.bias             | UNEXPECTED |  | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED |  | 
mask_predictions.LayerNorm.weight       | UNEXPECTED |  | 
mask_predictions.dense.weight           | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED |  | 
lm_predictions.lm_head.bias             | UNEXPECTED |  | 
mask_predictions.classifier.bias        | UNEXPECTED |  | 
mask_predictions.LayerNorm.bias         | UNEXPECTED |  | 
mask_predictions.classifier.weight      | UNEXPECTED |  | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


fold 3 | 3131 train, 780 val | 1173 steps
  val rows [ 0  5 19] ... [3901 3902 3903]
  ep0 step  40  val 0.5608  best 0.5608  48s  *
  ep0 step  80  val 0.6157  best 0.5608  97s
  ep0 step 120  val 0.5694  best 0.5608  145s
  ep0 step 160  val 0.5422  best 0.5422  193s  *
  ep0 step 200  val 0.5355  best 0.5355  242s  *
  ep0 step 240  val 0.7005  best 0.5355  290s
  ep0 step 280  val 0.6034  best 0.5355  338s
  ep0 step 320  val 0.5368  best 0.5355  386s
  ep0 step 360  val 0.5591  best 0.5355  435s
  ep0 END      val 0.6013  best 0.5355  477s
  ep1 step 400  val 0.5240  best 0.5240  501s  *
  ep1 step 440  val 0.5644  best 0.5240  550s
  ep1 step 480  val 0.5310  best 0.5240  598s
  ep1 step 520  val 0.5052  best 0.5052  646s  *
  ep1 step 560  val 0.6052  best 0.5052  694s
  ep1 step 600  val 0.5466  best 0.5052  743s
  ep1 step 640  val 0.4971  best 0.4971  791s  *
  ep1 step 680  val 0.5525  best 0.4971  839s
  ep1 step 720  val 0.5941  best 0.4971  888s
  ep1 step 760  val 0.5486

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

DebertaV2Model LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
mask_predictions.dense.bias             | UNEXPECTED |  | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED |  | 
mask_predictions.LayerNorm.weight       | UNEXPECTED |  | 
mask_predictions.dense.weight           | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED |  | 
lm_predictions.lm_head.bias             | UNEXPECTED |  | 
mask_predictions.classifier.bias        | UNEXPECTED |  | 
mask_predictions.LayerNorm.bias         | UNEXPECTED |  | 
mask_predictions.classifier.weight      | UNEXPECTED |  | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


fold 4 | 3133 train, 778 val | 1173 steps
  val rows [ 6 11 13] ... [3893 3898 3905]
  ep0 step  40  val 0.5824  best 0.5824  48s  *
  ep0 step  80  val 0.5793  best 0.5793  96s  *
  ep0 step 120  val 0.5716  best 0.5716  144s  *
  ep0 step 160  val 0.5784  best 0.5716  192s
  ep0 step 200  val 0.5913  best 0.5716  239s
  ep0 step 240  val 0.6974  best 0.5716  287s
  ep0 step 280  val 0.5931  best 0.5716  335s
  ep0 step 320  val 0.5147  best 0.5147  384s  *
  ep0 step 360  val 0.6540  best 0.5147  432s
  ep0 END      val 0.5541  best 0.5147  474s
  ep1 step 400  val 0.5527  best 0.5147  498s
  ep1 step 440  val 0.5239  best 0.5147  546s
  ep1 step 480  val 0.5310  best 0.5147  594s
  ep1 step 520  val 0.5631  best 0.5147  642s
  ep1 step 560  val 0.5907  best 0.5147  690s
  ep1 step 600  val 0.5557  best 0.5147  738s
  ep1 step 640  val 0.5011  best 0.5011  786s  *
  ep1 step 680  val 0.5241  best 0.5011  835s
  ep1 step 720  val 0.5683  best 0.5011  884s
  ep1 step 760  val 0.5064  b

In [53]:
D = '/kaggle/working/saved'
np.save(f'{D}/oof_deberta.npy', oof_deberta)
train.to_csv(f'{D}/train_5folds.csv', index=False)
np.save(f'{D}/oof_tfidf_ridge.npy', oof_tfidf)
np.save(f'{D}/oof_feat_ridge.npy', oof_r)
np.save(f'{D}/oof_feat_lgb.npy', oof_l)
np.save(f'{D}/oof_feat_xgb.npy', oof_x)
X_feat.to_csv(f'{D}/features_106.csv', index=False)
print(sorted(os.listdir(D)))

['deberta_fold0_idx.npy', 'deberta_fold0_preds.npy', 'deberta_fold1_idx.npy', 'deberta_fold1_preds.npy', 'deberta_fold2_idx.npy', 'deberta_fold2_preds.npy', 'deberta_fold3_idx.npy', 'deberta_fold3_preds.npy', 'deberta_fold4_idx.npy', 'deberta_fold4_preds.npy', 'features_106.csv', 'oof_deberta.npy', 'oof_feat_lgb.npy', 'oof_feat_ridge.npy', 'oof_feat_xgb.npy', 'oof_tfidf_ridge.npy', 'train_5folds.csv']


In [54]:
Y = train[TARGETS].values
mods  = [oof_deberta, oof_tfidf, oof_r, oof_l, oof_x]
names = ['deberta', 'tfidf', 'feat_ridge', 'feat_lgb', 'feat_xgb']

print('individual models')
print('-' * 34)
for n, m in zip(names, mods):
    print(f'  {n:<14} {mcrmse(Y, m):.4f}')

META = np.hstack(mods)
print('\nmeta matrix:', META.shape)

def run_stack(alpha):
    oof = np.zeros_like(Y)
    for f in range(5):
        tr = train.index[train.fold != f].values
        va = train.index[train.fold == f].values
        for j in range(6):
            m = Ridge(alpha=alpha)
            m.fit(META[tr], Y[tr, j])
            oof[va, j] = m.predict(META[va])
    return np.clip(oof, 1, 5)

print('\nstacked (Ridge meta-model)')
print('-' * 34)
best_s, best_oof, best_a = 9, None, None
for a in [1, 5, 20, 50, 100, 200]:
    o = run_stack(a)
    s = mcrmse(Y, o)
    print(f'  alpha {a:>4}      {s:.4f}')
    if s < best_s:
        best_s, best_oof, best_a = s, o, a

print(f'\nBEST STACK: {best_s:.4f}  (alpha={best_a})')
_, c = mcrmse(Y, best_oof, per_column=True)
for n, v in zip(TARGETS, c):
    print(f'  {n:<14} {v:.4f}')
np.save(f'{D}/oof_final_stack.npy', best_oof)

individual models
----------------------------------
  deberta        0.4881
  tfidf          0.5288
  feat_ridge     0.5326
  feat_lgb       0.5330
  feat_xgb       0.5329

meta matrix: (3911, 30)

stacked (Ridge meta-model)
----------------------------------
  alpha    1      0.4755
  alpha    5      0.4740
  alpha   20      0.4732
  alpha   50      0.4733
  alpha  100      0.4737
  alpha  200      0.4748

BEST STACK: 0.4732  (alpha=20)
  cohesion       0.5005
  syntax         0.4591
  vocabulary     0.4254
  phraseology    0.4712
  grammar        0.5120
  conventions    0.4713


## CFG768

In [55]:
class CFG768:
    model_name    = 'microsoft/deberta-v3-base'
    max_len       = 768
    batch_size    = 4
    valid_bs      = 8
    epochs        = 3
    lr            = 2e-5
    head_lr       = 2e-5
    weight_decay  = 0.01
    warmup_ratio  = 0.0
    grad_accum    = 2
    max_grad_norm = 1000.0
    eval_every    = 40
    n_folds       = 5
    seed          = 42

print('CFG768  max_len:', CFG768.max_len,
      '| eff batch:', CFG768.batch_size * CFG768.grad_accum)
print('CFG     max_len:', CFG.max_len,
      '| eff batch:', CFG.batch_size * CFG.grad_accum)
assert CFG768.max_len == 768 and CFG.max_len == 512
print('both configs OK')

CFG768  max_len: 768 | eff batch: 8
CFG     max_len: 512 | eff batch: 8
both configs OK


## Train all 5 folds at 768

In [56]:
gc.collect(); torch.cuda.empty_cache()
print('GPU free:', f'{torch.cuda.mem_get_info()[0]/1e9:.2f} GB')

oof_768 = np.full((len(train), 6), np.nan)
scores_768, t0 = [], time.time()

for f in range(5):
    print(f'\n===== FOLD {f} @ 768 =====')
    s, p, idx = train_fold(f, CFG768)

    assert len(idx) == len(p)
    assert np.isnan(oof_768[idx]).all(), f'fold {f} overwriting!'
    oof_768[idx] = p
    scores_768.append(s)

    np.save(f'/kaggle/working/saved/deberta768_fold{f}_preds.npy', p)
    np.save(f'/kaggle/working/saved/deberta768_fold{f}_idx.npy', idx)
    np.save('/kaggle/working/saved/oof_deberta768.npy', oof_768)

    filled = int((~np.isnan(oof_768[:, 0])).sum())
    done = (time.time() - t0) / 60
    print(f'>>> fold {f}: {s:.4f} | 512 was {[0.4834,0.4742,0.4862,0.4969,0.5018][f]:.4f} '
          f'| filled {filled}/{len(train)} | {done:.0f} min | '
          f'~{done/(f+1)*(5-f-1):.0f} min left')

assert not np.isnan(oof_768).any()
oof_768 = np.clip(oof_768, 1, 5)
cv768, c768 = mcrmse(train[TARGETS].values, oof_768, per_column=True)
_, c512 = mcrmse(train[TARGETS].values, oof_deberta, per_column=True)

print(f'\nfold scores 768: {[round(x,4) for x in scores_768]}')
print(f'mean of folds  : {np.mean(scores_768):.4f}')
print(f'OOF CV @ 768   : {cv768:.4f}')
print(f'OOF CV @ 512   : 0.4886')
print(f'\n{"target":<14}{"512":>9}{"768":>9}{"gain":>9}')
for n, a, b in zip(TARGETS, c512, c768):
    print(f'{n:<14}{a:>9.4f}{b:>9.4f}{a-b:>+9.4f}')
np.save('/kaggle/working/saved/oof_deberta768.npy', oof_768)

GPU free: 15.46 GB

===== FOLD 0 @ 768 =====


Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

DebertaV2Model LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
mask_predictions.dense.bias             | UNEXPECTED |  | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED |  | 
mask_predictions.LayerNorm.weight       | UNEXPECTED |  | 
mask_predictions.dense.weight           | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED |  | 
lm_predictions.lm_head.bias             | UNEXPECTED |  | 
mask_predictions.classifier.bias        | UNEXPECTED |  | 
mask_predictions.LayerNorm.bias         | UNEXPECTED |  | 
mask_predictions.classifier.weight      | UNEXPECTED |  | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


fold 0 | 3130 train, 781 val | 1173 steps
  val rows [ 1  7 12] ... [3895 3900 3910]
  ep0 step  40  val 0.5787  best 0.5787  79s  *
  ep0 step  80  val 0.5981  best 0.5787  157s
  ep0 step 120  val 0.5691  best 0.5691  236s  *
  ep0 step 160  val 0.6099  best 0.5691  315s
  ep0 step 200  val 0.5655  best 0.5655  396s  *
  ep0 step 240  val 0.5113  best 0.5113  476s  *
  ep0 step 280  val 0.5267  best 0.5113  556s
  ep0 step 320  val 0.5986  best 0.5113  636s
  ep0 step 360  val 0.5361  best 0.5113  715s
  ep0 END      val 0.6879  best 0.5113  783s
  ep1 step 400  val 0.5013  best 0.5013  826s  *
  ep1 step 440  val 0.5427  best 0.5013  907s
  ep1 step 480  val 0.5345  best 0.5013  987s
  ep1 step 520  val 0.5353  best 0.5013  1066s
  ep1 step 560  val 0.4843  best 0.4843  1146s  *
  ep1 step 600  val 0.5035  best 0.4843  1223s
  ep1 step 640  val 0.6143  best 0.4843  1302s
  ep1 step 680  val 0.5225  best 0.4843  1382s
  ep1 step 720  val 0.5134  best 0.4843  1461s
  ep1 step 760  val

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

...

DebertaV2Model LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
mask_predictions.dense.bias             | UNEXPECTED |  | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED |  | 
mask_predictions.LayerNorm.weight       | UNEXPECTED |  | 
mask_predictions.dense.weight           | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED |  | 
lm_predictions.lm_head.bias             | UNEXPECTED |  | 
mask_predictions.classifier.bias        | UNEXPECTED |  | 
mask_predictions.LayerNorm.bias         | UNEXPECTED |  | 
mask_predictions.classifier.weight      | UNEXPECTED |  | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


fold 3 | 3131 train, 780 val | 1173 steps
  val rows [ 0  5 19] ... [3901 3902 3903]
  ep0 step  40  val 0.5874  best 0.5874  80s  *
  ep0 step  80  val 0.5950  best 0.5874  158s
  ep0 step 120  val 0.5789  best 0.5789  238s  *
  ep0 step 160  val 0.5504  best 0.5504  317s  *
  ep0 step 200  val 0.5209  best 0.5209  396s  *
  ep0 step 240  val 0.6449  best 0.5209  475s
  ep0 step 280  val 0.6116  best 0.5209  554s
  ep0 step 320  val 0.5642  best 0.5209  631s
  ep0 step 360  val 0.5517  best 0.5209  710s
  ep0 END      val 0.5885  best 0.5209  778s
  ep1 step 400  val 0.5236  best 0.5209  821s
  ep1 step 440  val 0.5176  best 0.5176  900s  *
  ep1 step 480  val 0.5423  best 0.5176  979s
  ep1 step 520  val 0.4903  best 0.4903  1057s  *
  ep1 step 560  val 0.6064  best 0.4903  1137s
  ep1 step 600  val 0.5498  best 0.4903  1215s
  ep1 step 640  val 0.4964  best 0.4903  1293s
  ep1 step 680  val 0.5370  best 0.4903  1373s
  ep1 step 720  val 0.5837  best 0.4903  1450s
  ep1 step 760  val

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

DebertaV2Model LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
mask_predictions.dense.bias             | UNEXPECTED |  | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED |  | 
mask_predictions.LayerNorm.weight       | UNEXPECTED |  | 
mask_predictions.dense.weight           | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED |  | 
lm_predictions.lm_head.bias             | UNEXPECTED |  | 
mask_predictions.classifier.bias        | UNEXPECTED |  | 
mask_predictions.LayerNorm.bias         | UNEXPECTED |  | 
mask_predictions.classifier.weight      | UNEXPECTED |  | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


fold 4 | 3133 train, 778 val | 1173 steps
  val rows [ 6 11 13] ... [3893 3898 3905]
  ep0 step  40  val 0.5596  best 0.5596  77s  *
  ep0 step  80  val 0.5959  best 0.5596  155s
  ep0 step 120  val 0.6040  best 0.5596  234s
  ep0 step 160  val 0.5244  best 0.5244  315s  *
  ep0 step 200  val 0.5102  best 0.5102  397s  *
  ep0 step 240  val 0.5997  best 0.5102  476s
  ep0 step 280  val 0.5959  best 0.5102  557s
  ep0 step 320  val 0.4998  best 0.4998  637s  *
  ep0 step 360  val 0.6171  best 0.4998  718s
  ep0 END      val 0.5404  best 0.4998  789s
  ep1 step 400  val 0.5487  best 0.4998  832s
  ep1 step 440  val 0.5229  best 0.4998  911s
  ep1 step 480  val 0.5179  best 0.4998  995s
  ep1 step 520  val 0.5836  best 0.4998  1074s
  ep1 step 560  val 0.6469  best 0.4998  1153s
  ep1 step 600  val 0.5618  best 0.4998  1232s
  ep1 step 640  val 0.5223  best 0.4998  1308s
  ep1 step 680  val 0.5336  best 0.4998  1387s
  ep1 step 720  val 0.5787  best 0.4998  1469s
  ep1 step 760  val 0.521

# Meta-model comparison

In [57]:
from sklearn.linear_model import Ridge, Lasso, ElasticNet, LinearRegression, HuberRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from sklearn.cross_decomposition import PLSRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from scipy.optimize import nnls
import lightgbm as lgb

mods  = [oof_deberta, oof_768, oof_tfidf, oof_r, oof_l, oof_x]
names = ['deberta512','deberta768','tfidf','feat_ridge','feat_lgb','feat_xgb']
META  = np.hstack(mods)
print('meta matrix:', META.shape)
print('512 vs 768 correlation:',
      f'{np.corrcoef(oof_deberta.ravel(), oof_768.ravel())[0,1]:.4f}')

def run_meta(make_model, tag):
    oof = np.zeros_like(Y)
    for f in range(5):
        tr = train.index[train.fold != f].values
        va = train.index[train.fold == f].values
        for j in range(6):
            m = make_model(); m.fit(META[tr], Y[tr, j])
            oof[va, j] = np.ravel(m.predict(META[va]))
    oof = np.clip(oof, 1, 5); s = mcrmse(Y, oof)
    print(f'  {tag:<32} {s:.4f}')
    return s, oof

def run_nnls():
    oof = np.zeros_like(Y)
    for f in range(5):
        tr = train.index[train.fold != f].values
        va = train.index[train.fold == f].values
        for j in range(6):
            w, _ = nnls(META[tr], Y[tr, j])
            oof[va, j] = META[va] @ w
    oof = np.clip(oof, 1, 5); s = mcrmse(Y, oof)
    print(f'  {"NNLS (non-negative)":<32} {s:.4f}')
    return s, oof

print('\nMETA-MODEL COMPARISON')
print('=' * 50)
res = {}
res['average'] = (mcrmse(Y, np.clip(np.mean(mods, 0), 1, 5)),
                  np.clip(np.mean(mods, 0), 1, 5))
print(f'  {"simple average":<32} {res["average"][0]:.4f}')
res['ols']  = run_meta(lambda: LinearRegression(), 'LinearRegression (no penalty)')
res['nnls'] = run_nnls()

best_a, best_s = None, 9
for a in [1, 5, 20, 50, 100, 200, 400]:
    s, o = run_meta(lambda a=a: Ridge(alpha=a), f'Ridge alpha={a}')
    if s < best_s: best_s, best_a, res['ridge'] = s, a, (s, o)
print(f'  -> best Ridge alpha={best_a}')

res['lasso'] = run_meta(lambda: Lasso(alpha=0.01, max_iter=5000), 'Lasso')
res['enet']  = run_meta(lambda: ElasticNet(alpha=0.01, l1_ratio=0.5, max_iter=5000), 'ElasticNet')
res['huber'] = run_meta(lambda: make_pipeline(StandardScaler(),
                        HuberRegressor(alpha=1.0, max_iter=500)), 'Huber')
res['pls']   = run_meta(lambda: PLSRegression(n_components=6), 'PLS')
res['svr']   = run_meta(lambda: make_pipeline(StandardScaler(),
                        SVR(C=1.0, epsilon=0.1)), 'SVR rbf')
res['rf']    = run_meta(lambda: RandomForestRegressor(n_estimators=200, max_depth=4,
                        min_samples_leaf=50, n_jobs=-1, random_state=42), 'RandomForest')
res['lgbm']  = run_meta(lambda: lgb.LGBMRegressor(n_estimators=150, learning_rate=0.03,
                        num_leaves=7, min_child_samples=80, reg_lambda=10.0,
                        verbose=-1, random_state=42), 'LightGBM')

print('=' * 50)
ranked = sorted(res.items(), key=lambda kv: kv[1][0])
print('\nRANKING')
for k, (s, _) in ranked:
    print(f'  {k:<14} {s:.4f}')
bn, (bs, boof) = ranked[0]
print(f'\nWINNER: {bn} {bs:.4f}')
print('within 0.002:', [k for k, (s, _) in ranked if s - bs <= 0.002])
print('-> if tied, prefer Ridge (simplest, validated at every stage)')

_, cols = mcrmse(Y, boof, per_column=True)
print('\nfinal per-column:')
for n, v in zip(TARGETS, cols): print(f'  {n:<14} {v:.4f}')
np.save(f'{D}/oof_final_stack.npy', boof)

meta matrix: (3911, 36)
512 vs 768 correlation: 0.9666

META-MODEL COMPARISON
  simple average                   0.4855
  LinearRegression (no penalty)    0.4856
  NNLS (non-negative)              0.4747
  Ridge alpha=1                    0.4786
  Ridge alpha=5                    0.4740
  Ridge alpha=20                   0.4720
  Ridge alpha=50                   0.4716
  Ridge alpha=100                  0.4718
  Ridge alpha=200                  0.4726
  Ridge alpha=400                  0.4741
  -> best Ridge alpha=50
  Lasso                            0.4756
  ElasticNet                       0.4735
  Huber                            0.4840
  PLS                              0.4728
  SVR rbf                          0.4806
  RandomForest                     0.4759
  LightGBM                         0.4751

RANKING
  ridge          0.4716
  pls            0.4728
  enet           0.4735
  nnls           0.4747
  lgbm           0.4751
  lasso          0.4756
  rf             0.4759
  svr 

# Ablation: what did 768 buy

In [58]:
def stack_subset(model_list, tag):
    M = np.hstack(model_list)
    best = 9
    for a in [1, 5, 20, 50, 100, 200, 400]:
        oof = np.zeros_like(Y)
        for f in range(5):
            tr = train.index[train.fold != f].values
            va = train.index[train.fold == f].values
            for j in range(6):
                m = Ridge(alpha=a).fit(M[tr], Y[tr, j])
                oof[va, j] = m.predict(M[va])
        best = min(best, mcrmse(Y, np.clip(oof, 1, 5)))
    print(f'  {tag:<34} {best:.4f}')
    return best

print('ABLATION — what each component adds')
print('=' * 50)
s1 = stack_subset([oof_tfidf, oof_r, oof_l, oof_x], 'classical only (4 models)')
s2 = stack_subset([oof_deberta, oof_tfidf, oof_r, oof_l, oof_x], '+ deberta512')
s3 = stack_subset(mods, '+ deberta768 (all 6)')
print('=' * 50)
print(f'  deberta512 added: {s1-s2:+.4f}')
print(f'  deberta768 added: {s2-s3:+.4f}')

ABLATION — what each component adds
  classical only (4 models)          0.5042
  + deberta512                       0.4732
  + deberta768 (all 6)               0.4716
  deberta512 added: +0.0309
  deberta768 added: +0.0016


# Error-Analysis

In [59]:
best_oof = boof                      # winner from Cell B
Y = train[TARGETS].values
err_per_target = np.abs(best_oof - Y)
err = err_per_target.mean(axis=1)

train['abs_err']   = err
train['pred_mean'] = best_oof.mean(axis=1)
train['true_mean'] = Y.mean(axis=1)
train['n_words']   = train['full_text'].str.split().str.len()
train['n_sents']   = train['full_text'].str.count(r'[.!?]') + 1
train['lower_start'] = train['full_text'].apply(
    lambda t: np.mean([s.strip()[:1].islower()
                       for s in t.split('.') if s.strip()]) if t.strip() else 0)

print('=' * 62)
print('1. OVERALL')
print('=' * 62)
print(f'  MCRMSE           : {mcrmse(Y, best_oof):.4f}')
print(f'  mean abs error   : {err.mean():.4f}')
print(f'  median abs error : {np.median(err):.4f}')
print(f'  90th percentile  : {np.percentile(err, 90):.4f}')
print(f'  worst            : {err.max():.4f}')

print('\n' + '=' * 62)
print('2. WHICH TARGET IS HARDEST')
print('=' * 62)
_, cols = mcrmse(Y, best_oof, per_column=True)
base_std = train[TARGETS].std(ddof=0).values
print(f'  {"target":<14}{"RMSE":>9}{"baseline":>10}{"% gain":>9}')
for n, v, b in zip(TARGETS, cols, base_std):
    print(f'  {n:<14}{v:>9.4f}{b:>10.4f}{100*(b-v)/b:>8.1f}%')

print('\n' + '=' * 62)
print('3. REGRESSION TO THE MEAN')
print('=' * 62)
print(f'  {"true band":<14}{"n":>6}{"MAE":>9}{"bias":>10}')
for lo, hi in [(1,2),(2,2.5),(2.5,3),(3,3.5),(3.5,4),(4,4.5),(4.5,5.1)]:
    m = ((train.true_mean >= lo) & (train.true_mean < hi)).values
    if m.sum() >= 5:
        bias = (train.pred_mean.values[m] - train.true_mean.values[m]).mean()
        print(f'  [{lo:.1f},{hi:.1f}){"":<6}{m.sum():>6}{err[m].mean():>9.4f}{bias:>+10.4f}')
print(f'\n  std of true : {train.true_mean.std():.4f}')
print(f'  std of pred : {train.pred_mean.std():.4f}')
print(f'  ratio       : {train.pred_mean.std()/train.true_mean.std():.4f}'
      '   <1 = under-dispersed (expected under MSE loss)')

print('\n' + '=' * 62)
print('4. ERROR BY ESSAY LENGTH')
print('=' * 62)
print(f'  {"words":<14}{"n":>6}{"MAE":>9}{"bias":>10}')
for lo, hi in [(0,150),(150,250),(250,350),(350,500),(500,700),(700,9999)]:
    m = ((train.n_words >= lo) & (train.n_words < hi)).values
    if m.sum() >= 5:
        bias = (train.pred_mean.values[m] - train.true_mean.values[m]).mean()
        print(f'  {lo}-{hi if hi<9999 else "+":<9}{m.sum():>6}{err[m].mean():>9.4f}{bias:>+10.4f}')

print('\n' + '=' * 62)
print('5. TRUNCATION EFFECT (essays over 512 tokens)')
print('=' * 62)
long_m = (train.n_words > 450).values
print(f'  short essays (<=450 words) n={(~long_m).sum():>4}  MAE={err[~long_m].mean():.4f}')
print(f'  long  essays (> 450 words) n={long_m.sum():>4}  MAE={err[long_m].mean():.4f}')
print('\n  per-target MAE, long vs short:')
for j, t in enumerate(TARGETS):
    s_, l_ = err_per_target[~long_m, j].mean(), err_per_target[long_m, j].mean()
    print(f'    {t:<14} short {s_:.4f}   long {l_:.4f}   diff {l_-s_:+.4f}')

print('\n' + '=' * 62)
print('6. DO MODELS DISAGREE WHERE ERROR IS HIGH?')
print('=' * 62)
disagree = np.std([m.mean(axis=1) for m in mods], axis=0)
print(f'  correlation(model disagreement, error): '
      f'{np.corrcoef(disagree, err)[0,1]:.4f}')
q = pd.qcut(disagree, 4, labels=['low','med','high','very high'])
for lab in ['low','med','high','very high']:
    m = (q == lab).values
    print(f'    disagreement {lab:<10} n={m.sum():>4}  MAE={err[m].mean():.4f}')

print('\n' + '=' * 62)
print('7. WORST 5 PREDICTIONS')
print('=' * 62)
for i in train.nlargest(5, 'abs_err').index:
    print(f'\n  {train.loc[i,"text_id"]}  err={err[i]:.2f}  '
          f'words={train.loc[i,"n_words"]}  lower_starts='
          f'{train.loc[i,"lower_start"]:.2f}')
    print(f'    true: {Y[i].round(1)}')
    print(f'    pred: {best_oof[i].round(1)}')
    print(f'    text: {train.loc[i,"full_text"][:150]}...')

print('\n' + '=' * 62)
print('8. BEST 3 PREDICTIONS (for contrast)')
print('=' * 62)
for i in train.nsmallest(3, 'abs_err').index:
    print(f'  {train.loc[i,"text_id"]}  err={err[i]:.3f}  '
          f'true={Y[i].round(1)}  words={train.loc[i,"n_words"]}')

train.drop(columns=['abs_err','pred_mean','true_mean','n_words',
                    'n_sents','lower_start'], inplace=True, errors='ignore')

1. OVERALL
  MCRMSE           : 0.4716
  mean abs error   : 0.3783
  median abs error : 0.3502
  90th percentile  : 0.5802
  worst            : 1.3705

2. WHICH TARGET IS HARDEST
  target             RMSE  baseline   % gain
  cohesion         0.4982    0.6625    24.8%
  syntax           0.4578    0.6443    28.9%
  vocabulary       0.4247    0.5831    27.2%
  phraseology      0.4704    0.6559    28.3%
  grammar          0.5108    0.6998    27.0%
  conventions      0.4679    0.6714    30.3%

3. REGRESSION TO THE MEAN
  true band          n      MAE      bias
  [1.0,2.0)          46   0.4908   +0.4409
  [2.0,2.5)         444   0.3894   +0.2475
  [2.5,3.0)        1138   0.3729   +0.1691
  [3.0,3.5)        1246   0.3466   -0.0292
  [3.5,4.0)         795   0.3666   -0.1904
  [4.0,4.5)         189   0.5085   -0.4736
  [4.5,5.1)          53   0.7619   -0.7557

  std of true : 0.5609
  std of pred : 0.4413
  ratio       : 0.7868   <1 = under-dispersed (expected under MSE loss)

4. ERROR BY ESSA

AttributeError: 'numpy.ndarray' object has no attribute 'values'